# Amazon ML — deadline upgrade, cached training + submission

Keep your submitted **0.84** result as the fallback. This notebook attempts one measured improvement:
- **30,000 training businesses**, 2,000 tuning businesses, and 2,000 fresh evaluation businesses.
- Same fixed E5 model and 19 comparison features; more labeled examples for the matcher.
- Compare standard search (64 cells / 200 intermediate neighbors) with stronger search (128 / 400), always **20 final candidates**.
- Choose search and cutoff on tuning data and estimated runtime. Compare the chosen new pipeline to the saved old model on the same fresh evaluation businesses.
- Start full test prediction automatically **only if evaluation improves and the estimated time fits the target**.

## Inputs and settings
Use a new Kaggle notebook. Attach **(1) the original challenge dataset and (2) the full saved output of the previous notebook**. The second input needs `full_pool_embedding_v1/vectors.f16`, `records.sqlite`, country `.faiss` files, `sample.tsv`, `matcher.pkl`, and metadata. The small results ZIP alone is insufficient.

Enable GPU + Internet. Do not copy the large training caches to working storage: they are read directly from the attached input. Training target embeddings/indexes are reused. Test embeddings still need to be generated.

The planned finish is **27 September 2026, 10:30 p.m. IST**, leaving time before the stated 11:59 p.m. deadline. Estimates include a margin but are not guarantees. Use **Save Version → Save & Run All**. Expect the evaluation stage to finish much sooner than full submission prediction; watch its report.


In [ ]:
%pip install -q "numpy==2.0.2" "pandas==2.3.3" "scikit-learn==1.6.1" "faiss-cpu==1.11.0" "transformers==5.0.0" safetensors sentencepiece


In [ ]:
from pathlib import Path
import os,sys,json,subprocess,shutil
DATA=Path('/kaggle/input/datasets/spoortimorabad/ml-challenge-dataset/student_resource/dataset')
# Leave blank for automatic discovery. If needed, set to the attached full_pool_embedding_v1 folder.
CACHE_DIR=''
ROOT=Path('/kaggle/working/ml_deadline_upgrade_v2')
CODE_DIR=ROOT/'code'
UPGRADE_WORK=ROOT/'training_upgrade'
SUBMISSION_WORK=ROOT/'submission'
TARGET_FINISH='2026-09-27T22:30:00+05:30'
BATCH_SIZE=64
CODE_DIR.mkdir(parents=True,exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM']='false'
import torch
assert torch.cuda.is_available(),'Enable a GPU in notebook settings.'
assert (DATA/'train/train_source1.tsv').exists(),'Update DATA to the challenge dataset path.'
assert (DATA.parent/'utils/validate_submission.py').exists(),'The original challenge utils folder is required.'
print('GPU:',torch.cuda.get_device_name(0))
print('Free working disk GiB:',round(shutil.disk_usage(ROOT).free/2**30,2))


In [ ]:
MODULES={'embedding_core.py': 'import os, json, time, gc, sqlite3, hashlib, shutil, pickle, re, unicodedata\nfrom pathlib import Path\nfrom functools import lru_cache\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom threadpoolctl import threadpool_limits\nCOLS = [\'entity_id\', \'business_name\', \'business_address\', \'country\']\n\nLEGAL = set(\'inc incorporated llc llp ltd limited pvt private corp corporation company co plc sarl sas eurl\'.split())\n\nADDR_ALIAS = {\'road\': \'rd\', \'street\': \'st\', \'avenue\': \'ave\', \'drive\': \'dr\', \'lane\': \'ln\', \'boulevard\': \'blvd\', \'apartment\': \'apt\', \'suite\': \'ste\'}\n\nFEATURES = [\'name_exact\', \'core_name_exact\', \'name_trigram_jaccard\', \'name_token_jaccard\', \'name_token_containment\', \'name_length_ratio\', \'address_exact\', \'address_trigram_jaccard\', \'address_token_jaccard\', \'address_token_containment\', \'address_missing\', \'number_jaccard\', \'number_disjoint\', \'unit_equal\', \'unit_conflict\', \'country_equal\', \'name_token_count\', \'candidate_name_token_count\']\n\ndef log(s):\n    print(s, flush=True)\n\ndef norm(s):\n    s = unicodedata.normalize(\'NFKC\', str(s)).casefold()\n    return \' \'.join(re.findall(\'[^\\\\W_]+\', s))\n\ndef canonical_address(s):\n    return \' \'.join((ADDR_ALIAS.get(t, t) for t in norm(s).split() if t not in {\'null\', \'none\'}))\n\ndef grams(s):\n    return frozenset((s[i:i + 3] for i in range(max(0, len(s) - 2))))\n\ndef unit_id(s):\n    m = re.search(\'\\\\b(?:unit|suite|ste|apt|apartment|flat)\\\\s*(?:no\\\\.?\\\\s*)?[:#-]?\\\\s*([\\\\w]+)\', unicodedata.normalize(\'NFKC\', s).casefold())\n    return m.group(1) if m else \'\'\n\n@lru_cache(maxsize=30000)\ndef prepared(name, address, country):\n    n = norm(name)\n    a = canonical_address(address)\n    nt, at = (frozenset(n.split()), frozenset(a.split()))\n    core = tuple(sorted((t for t in nt if t not in LEGAL)))\n    nums = frozenset(re.findall(\'\\\\d+[a-z]*\', a))\n    return (n, a, nt, at, core, nums, grams(n), grams(a), unit_id(address), norm(country))\n\ndef jaccard(a, b):\n    return len(a & b) / len(a | b) if a or b else 0.0\n\ndef containment(a, b):\n    return len(a & b) / min(len(a), len(b)) if a and b else 0.0\n\ndef features(left, right):\n    n, a, nt, at, c, nums, ng, ag, u, country = prepared(*left[1:])\n    m, b, mt, bt, d, other_nums, mg, bg, v, other_country = prepared(*right[1:])\n    return [float(bool(n) and n == m), float(bool(c) and c == d), jaccard(ng, mg), jaccard(nt, mt), containment(nt, mt), min(len(n), len(m)) / max(len(n), len(m), 1), float(bool(a) and a == b), jaccard(ag, bg), jaccard(at, bt), containment(at, bt), float(not a or not b), jaccard(nums, other_nums), float(bool(nums) and bool(other_nums) and (not nums & other_nums)), float(bool(u) and bool(v) and (u == v)), float(bool(u) and bool(v) and (u != v)), float(bool(country) and country == other_country), len(nt), len(mt)]\n\ndef read_chunks(path, cols=COLS):\n    return pd.read_csv(path, sep=\'\\t\', dtype=str, keep_default_na=False, usecols=cols, chunksize=100000)\n\ndef reproduce_reference_sample(data):\n    from sklearn.model_selection import train_test_split\n    rng = np.random.default_rng(2026)\n    sample = pd.DataFrame()\n    for chunk in read_chunks(data / \'train/train_source1.tsv\'):\n        chunk = chunk[COLS].copy()\n        chunk[\'_priority\'] = rng.random(len(chunk))\n        sample = pd.concat([sample, chunk], ignore_index=True).nsmallest(4000, \'_priority\')\n    sample = sample.drop(columns=\'_priority\').reset_index(drop=True)\n    assert len(sample) == 4000 and sample.entity_id.is_unique\n    ids = set(sample.entity_id)\n    truth = {}\n    for chunk in read_chunks(data / \'train/train_ground_truth.tsv\', [\'source1_entity_id\', \'matched_entity_ids\']):\n        for row in chunk[chunk.source1_entity_id.isin(ids)].itertuples(index=False):\n            truth[row.source1_entity_id] = set((x.strip() for x in row.matched_entity_ids.split(\',\') if x.strip()))\n    assert set(truth) == ids\n    strata = sample.country + \'|\' + sample.entity_id.map(lambda x: str(not truth[x]))\n    train_idx, held = train_test_split(np.arange(4000), test_size=0.25, random_state=2026, stratify=strata)\n    tune_idx, eval_idx = train_test_split(held, test_size=0.5, random_state=2027, stratify=strata.iloc[held])\n    sample[\'split\'] = \'train\'\n    sample.loc[tune_idx, \'split\'] = \'tune\'\n    sample.loc[eval_idx, \'split\'] = \'evaluation\'\n    return (sample, truth)\n\ndef clean_embedding_text(s):\n    return \' \'.join(unicodedata.normalize(\'NFKC\', str(s)).split())\n\ndef text_view(frame, view):\n    if view == \'name\':\n        return [\'query: \' + clean_embedding_text(x) for x in frame.business_name]\n    return [\'query: business name: \' + clean_embedding_text(n) + \'; address: \' + clean_embedding_text(a) for n, a in zip(frame.business_name, frame.business_address)]\n\nMODEL_ID=\'intfloat/multilingual-e5-small\'\nREVISION=\'614241f622f53c4eeff9890bdc4f31cfecc418b3\'\nDIM=384\nVERSION=\'full-pool-e5-v1\'\n\ndef atomic_json(path, value):\n    tmp=Path(str(path)+\'.tmp\')\n    tmp.write_text(json.dumps(value,indent=2))\n    os.replace(tmp,path)\n\ndef initialize(data,work):\n    work=Path(work); data=Path(data); work.mkdir(parents=True,exist_ok=True)\n    files=[data/f\'train/train_source{i}.tsv\' for i in (1,2,3)]+[data/\'train/train_ground_truth.tsv\']\n    for p in files:\n        assert p.exists(),f\'Missing dataset: {p}\'\n    config={\'version\':VERSION,\'model\':MODEL_ID,\'revision\':REVISION,\'dim\':DIM,\n            \'max_length\':256,\'text\':\'query: business name: NAME; address: ADDRESS\',\n            \'files\':[(str(p),p.stat().st_size) for p in files],\n            \'nlist_max\':2048,\'pq_m\':48,\'pq_bits\':8,\'nprobe\':64,\'overfetch\':200,\n            \'candidate_k\':20,\'sample_seed\':2026,\'sample_size\':4000}\n    cp=work/\'configuration.json\'\n    if cp.exists():\n        assert json.loads(cp.read_text())==json.loads(json.dumps(config)), \'Configuration changed. Use a new WORK folder.\'\n    else: atomic_json(cp,config)\n    log(f\'Free disk: {shutil.disk_usage(work).free/2**30:.1f} GiB\')\n    return data,work\n\nclass Encoder:\n    def __init__(self,batch_size=64):\n        import torch\n        from transformers import AutoTokenizer,AutoModel\n        assert torch.cuda.is_available(),\'Enable a GPU in Kaggle Settings, then restart/run all.\'\n        self.torch=torch; self.batch_size=batch_size\n        self.tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False)\n        self.model=AutoModel.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False,\n                                           use_safetensors=True).cuda().half().eval()\n        log(\'GPU: \'+torch.cuda.get_device_name(0))\n    def encode(self,frame):\n        texts=text_view(frame,\'combined\'); parts=[]; torch=self.torch\n        with torch.inference_mode():\n            for start in range(0,len(texts),self.batch_size):\n                inp=self.tokenizer(texts[start:start+self.batch_size],max_length=256,\n                                   padding=True,truncation=True,return_tensors=\'pt\').to(\'cuda\')\n                hidden=self.model(**inp).last_hidden_state.float()\n                mask=inp[\'attention_mask\'].unsqueeze(-1)\n                vec=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)\n                vec=torch.nn.functional.normalize(vec,p=2,dim=1)\n                parts.append(vec.cpu().numpy())\n        return np.concatenate(parts).astype(\'float32\') if parts else np.empty((0,DIM),np.float32)\n\ndef open_records(work):\n    db=sqlite3.connect(work/\'records.sqlite\')\n    db.execute(\'PRAGMA cache_size=-65536\')\n    db.execute(\'CREATE TABLE IF NOT EXISTS records (rid INTEGER PRIMARY KEY, entity_id TEXT, business_name TEXT, business_address TEXT, country TEXT)\')\n    db.execute(\'CREATE TABLE IF NOT EXISTS progress (source INTEGER PRIMARY KEY, done INTEGER, complete INTEGER)\')\n    db.commit()\n    return db\n\ndef encode_pool(data,work,encoder,split="train"):\n    """Vectors flushed before SQLite transaction commits; uncommitted rows overwritten on restart."""\n    manifest_path=work/\'pool_manifest.json\'\n    if not manifest_path.exists():\n        counts={}\n        for source in (2,3):\n            counts[str(source)]=sum(len(c) for c in read_chunks(data/f\'{split}/{split}_source{source}.tsv\',[\'entity_id\']))\n        atomic_json(manifest_path,{\'counts\':counts,\'total\':sum(counts.values())})\n    manifest=json.loads(manifest_path.read_text()); total=manifest[\'total\']\n    vp=work/\'vectors.f16\'; needed=total*DIM*2\n    db=open_records(work)\n    exists=vp.exists()\n    if not exists:\n        assert db.execute(\'SELECT COUNT(*) FROM records\').fetchone()[0]==0,\'Vectors missing but database has progress. Restore both or use new WORK.\'\n        # Includes space for metadata DB, compressed indexes, model cache and reserve.\n        assert shutil.disk_usage(work).free>needed+5*2**30, \'Need about 13 GiB free before full-pool encoding. Use a fresh notebook.\'\n    else: assert vp.stat().st_size==needed,\'Incomplete vector allocation. Use new WORK directory.\'\n    vectors=np.memmap(vp,mode=\'r+\' if exists else \'w+\',dtype=np.float16,shape=(total,DIM))\n    base=0; started=time.time(); encoded=0\n    for source in (2,3):\n        saved=db.execute(\'SELECT done,complete FROM progress WHERE source=?\',(source,)).fetchone()\n        done=saved[0] if saved else 0\n        if saved and saved[1]:\n            log(f\'Source {source}: already encoded {done:,} rows\'); base+=manifest[\'counts\'][str(source)]; continue\n        log(f\'Source {source}: resume at {done:,}; reading input from beginning to that position\')\n        seen=0\n        chunks=pd.read_csv(data/f\'{split}/{split}_source{source}.tsv\',sep=\'\\t\',dtype=str,\n                           keep_default_na=False,usecols=COLS,chunksize=10000)\n        for frame in chunks:\n            end=seen+len(frame)\n            if end<=done: seen=end; continue\n            frame=frame.iloc[max(0,done-seen):][COLS]\n            first=max(done,seen); batch=encoder.encode(frame)\n            assert batch.shape==(len(frame),DIM) and np.isfinite(batch).all()\n            vectors[base+first:base+end]=batch.astype(np.float16); vectors.flush()\n            rows=[(base+first+j,str(r.entity_id),str(r.business_name),str(r.business_address),norm(r.country))\n                  for j,r in enumerate(frame.itertuples(index=False))]\n            with db:\n                db.executemany(\'INSERT INTO records VALUES (?,?,?,?,?)\',rows)\n                db.execute(\'INSERT OR REPLACE INTO progress VALUES (?,?,0)\',(source,end))\n            encoded+=len(frame); seen=end; done=end\n            if end%100000==0:\n                rate=encoded/max(time.time()-started,1)\n                remaining=total-(base+end)\n                log(f\'Source {source}: {end:,}; {rate:.0f} rows/sec; remaining encoding ~{remaining/rate/60:.1f} min\')\n                assert shutil.disk_usage(work).free>1.5*2**30,\'Low disk. Progress saved; free space before resuming.\'\n        assert done==manifest[\'counts\'][str(source)]\n        with db: db.execute(\'INSERT OR REPLACE INTO progress VALUES (?,?,1)\',(source,done))\n        base+=done\n    assert db.execute(\'SELECT COUNT(*) FROM records\').fetchone()[0]==total\n    atomic_json(work/\'encoding_complete.json\',{\'total\':total})\n    db.close(); del vectors\n    log(f\'Encoding complete: {total:,} target records.\')\n\ndef build_indexes(work):\n    """IVF-PQ index per country. Completed countries and trained quantizers are reusable."""\n    import faiss\n    faiss.omp_set_num_threads(min(4,os.cpu_count() or 2))\n    total=json.loads((work/\'encoding_complete.json\').read_text())[\'total\']\n    vectors=np.memmap(work/\'vectors.f16\',mode=\'r\',dtype=np.float16,shape=(total,DIM))\n    db=open_records(work)\n    countries=[r[0] for r in db.execute(\'SELECT DISTINCT country FROM records\')]\n    catalog=[]\n    for country in sorted(countries):\n        token=hashlib.sha256(country.encode()).hexdigest()[:12]\n        path=work/f\'index_{token}.faiss\'; trained=work/f\'trained_{token}.faiss\'\n        count=db.execute(\'SELECT COUNT(*) FROM records WHERE country=?\',(country,)).fetchone()[0]\n        catalog.append({\'country\':country,\'file\':path.name,\'count\':count})\n        if path.exists():\n            check=faiss.read_index(str(path)); assert check.ntotal==count\n            del check; log(f\'{country}: completed index reused\'); continue\n        ids=np.fromiter((r[0] for r in db.execute(\'SELECT rid FROM records WHERE country=? ORDER BY rid\',(country,))),dtype=np.int64,count=count)\n        # At least 50 training vectors per IVF centroid, up to 100k training vectors.\n        nlist=min(2048,max(1,count//50))\n        if trained.exists(): index=faiss.read_index(str(trained))\n        elif count<10000:\n            index=faiss.IndexIDMap2(faiss.IndexFlatIP(DIM))\n        else:\n            index=faiss.IndexIVFPQ(faiss.IndexFlatIP(DIM),DIM,nlist,48,8,faiss.METRIC_INNER_PRODUCT)\n            index.cp.seed=2026; index.pq.cp.seed=2026\n            train_ids=np.random.default_rng(2026).choice(ids,size=min(count,110000),replace=False)\n            train_vectors=np.asarray(vectors[np.sort(train_ids)],dtype=np.float32)\n            faiss.normalize_L2(train_vectors)\n            log(f\'{country}: training IVF-PQ on {len(train_vectors):,} vectors; {nlist} cells\')\n            index.train(train_vectors); del train_vectors\n            faiss.write_index(index,str(trained)+\'.tmp\'); os.replace(str(trained)+\'.tmp\',trained)\n        started=time.time()\n        # An interrupted country is re-added from its saved trained quantizer.\n        for start in range(0,count,25000):\n            batch_ids=ids[start:start+25000]\n            x=np.asarray(vectors[batch_ids],dtype=np.float32); faiss.normalize_L2(x)\n            index.add_with_ids(x,batch_ids)\n            if start%250000==0: log(f\'{country}: indexed {min(start+25000,count):,}/{count:,}; {(time.time()-started)/60:.1f} min\')\n        assert index.ntotal==count\n        faiss.write_index(index,str(path)+\'.tmp\'); os.replace(str(path)+\'.tmp\',path)\n        del index,ids; gc.collect()\n    atomic_json(work/\'index_catalog.json\',catalog)\n    db.close(); del vectors\n    log(\'All country indexes ready.\')\n\ndef prepare_queries(data,work,encoder):\n    sample_path=work/\'sample.pkl\'\n    if sample_path.exists():\n        with sample_path.open(\'rb\') as f: sample,truth=pickle.load(f)\n    else:\n        sample,truth=reproduce_reference_sample(data)\n        with Path(str(sample_path)+\'.tmp\').open(\'wb\') as f: pickle.dump((sample,truth),f)\n        os.replace(str(sample_path)+\'.tmp\',sample_path)\n        sample.to_csv(work/\'sample.tsv\',sep=\'\\t\',index=False)\n    path=work/\'query_vectors.npy\'\n    if path.exists(): q=np.load(path)\n    else:\n        q=encoder.encode(sample)\n        with Path(str(path)+\'.tmp\').open(\'wb\') as f: np.save(f,q)\n        os.replace(str(path)+\'.tmp\',path)\n    assert q.shape==(len(sample),DIM)\n    log(f\'Sample splits: {sample.split.value_counts().to_dict()}\')\n    return sample,truth,q\n\ndef retrieve(work,sample,q,nprobe=64,overfetch=200,save_k=50):\n    import faiss\n    query_key=hashlib.sha256((\'|\'.join(sample.entity_id)).encode()).hexdigest()[:12]\n    cache=work/f\'retrieval_{query_key}_p{nprobe}_r{overfetch}_k{save_k}.npz\'\n    if cache.exists():\n        saved=np.load(cache); return saved[\'ids\'],saved[\'scores\']\n    total=json.loads((work/\'encoding_complete.json\').read_text())[\'total\']\n    vectors=np.memmap(work/\'vectors.f16\',mode=\'r\',dtype=np.float16,shape=(total,DIM))\n    ids=np.full((len(q),save_k),-1,np.int64); scores=np.full((len(q),save_k),-1,np.float32)\n    countries=sample.country.map(norm).to_numpy()\n    for item in json.loads((work/\'index_catalog.json\').read_text()):\n        qi=np.flatnonzero(countries==item[\'country\'])\n        if not len(qi): continue\n        index=faiss.read_index(str(work/item[\'file\']))\n        if hasattr(index,\'nprobe\'): index.nprobe=min(nprobe,index.nlist)\n        log(f"Searching {len(qi)} businesses in {item[\'country\']}: {index.ntotal:,} records")\n        for start in range(0,len(qi),64):\n            batch=qi[start:start+64]\n            _,approx=index.search(np.ascontiguousarray(q[batch],dtype=np.float32),overfetch)\n            for i,neighbors in zip(batch,approx):\n                neighbors=np.unique(neighbors[neighbors>=0])\n                if not len(neighbors): continue\n                x=np.asarray(vectors[neighbors],dtype=np.float32)\n                x/=np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12)\n                exact=x@q[i]\n                order=np.lexsort((neighbors,-exact))[:save_k]\n                chosen=neighbors[order]; k=len(chosen)\n                ids[i,:k]=chosen; scores[i,:k]=exact[order]\n        del index; gc.collect()\n    with Path(str(cache)+\'.tmp\').open(\'wb\') as f: np.savez_compressed(f,ids=ids,scores=scores)\n    os.replace(str(cache)+\'.tmp\',cache)\n    return ids,scores\n\ndef fetch_records(work,ids):\n    db=open_records(work); rows={}\n    valid=np.unique(ids[ids>=0])\n    for start in range(0,len(valid),800):\n        batch=[int(x) for x in valid[start:start+800]]\n        for r in db.execute(\'SELECT * FROM records WHERE rid IN (\'+\',\'.join(\'?\'*len(batch))+\')\',batch):\n            rows[r[0]]=tuple(r[1:])\n    db.close()\n    assert len(rows)==len(valid)\n    return rows\n\ndef retrieval_report(sample,truth,ids,records):\n    out=[]\n    for split in (\'train\',\'tune\',\'evaluation\'):\n        indices=np.flatnonzero(sample.split.to_numpy()==split)\n        if not len(indices): continue\n        for k in (10,20,50):\n            found=total=0; sizes=[]\n            for i in indices:\n                gt=truth[sample.iloc[i].entity_id]\n                pred={records[int(r)][0] for r in ids[i,:k] if r>=0}\n                found+=len(gt&pred); total+=len(gt); sizes.append(len(pred))\n            out.append(dict(split=split,k=k,found=found,true_links=total,\n                            recall=found/total if total else 1,mean_candidates=float(np.mean(sizes)),\n                            p95_candidates=float(np.percentile(sizes,95)),max_candidates=max(sizes)))\n    return pd.DataFrame(out)\n\nMATCH_FEATURES=FEATURES+[\'embedding_cosine\']\ndef make_pairs(sample,truth,ids,scores,records,k=20):\n    x=[]; y=[]; owners=[]; candidate_ids=[]\n    for i,left in enumerate(sample[COLS].itertuples(index=False,name=None)):\n        for rank,rid in enumerate(ids[i,:k]):\n            if rid<0: continue\n            right=records[int(rid)]\n            x.append(features(left,right)+[float(scores[i,rank])])\n            y.append(int(right[0] in truth[left[0]])); owners.append(i); candidate_ids.append(right[0])\n    return np.asarray(x,np.float32),np.asarray(y,np.uint8),np.asarray(owners,np.int32),candidate_ids\n\ndef metrics(sample,truth,owners,candidate_ids,prob,threshold,split):\n    selected=np.flatnonzero(sample.split.to_numpy()==split)\n    pred={int(i):set() for i in selected}\n    for i,c,p in zip(owners,candidate_ids,prob):\n        if int(i) in pred and p>=threshold: pred[int(i)].add(c)\n    scores=[]; tp=fp=fn=singletons=singleton_correct=0\n    for i in selected:\n        gt=truth[sample.iloc[i].entity_id]; pr=pred[int(i)]\n        a=len(gt&pr); b=len(pr-gt); c=len(gt-pr)\n        tp+=a; fp+=b; fn+=c\n        if not gt:\n            singletons+=1; singleton_correct+=int(not pr); scores.append(float(not pr))\n        else: scores.append(1.25*a/(1.25*a+b+.25*c))\n    return dict(macro_F05=float(np.mean(scores)),correct_accepted=tp,incorrect_accepted=fp,\n                missed_true_links=fn,singletons_correct=singleton_correct,singletons=singletons)\n\ndef fit_and_evaluate(work,sample,truth,ids,scores,records):\n    from importlib.metadata import version\n    x,y,owners,candidate_ids=make_pairs(sample,truth,ids,scores,records,k=20)\n    splits=sample.split.to_numpy()[owners]; train=splits==\'train\'\n    assert len(np.unique(y[train]))==2,\'Need positive and negative training pairs.\'\n    log(f\'Training new matcher: {train.sum():,} pairs, {y[train].sum():,} true matches\')\n    model=HistGradientBoostingClassifier(learning_rate=.08,max_iter=250,max_leaf_nodes=31,\n              l2_regularization=1.,early_stopping=False,random_state=2026)\n    with threadpool_limits(limits=4):\n        model.fit(x[train],y[train]); prob=model.predict_proba(x)[:,1]\n    # Only tuning businesses choose the threshold. Evaluation is read once afterwards.\n    tuning=[]\n    for t in np.linspace(.05,.95,91):\n        tuning.append({\'threshold\':float(t),**metrics(sample,truth,owners,candidate_ids,prob,t,\'tune\')})\n    best=max(tuning,key=lambda d:(d[\'macro_F05\'],d[\'threshold\']))\n    cutoff=best[\'threshold\']\n    evaluation=metrics(sample,truth,owners,candidate_ids,prob,cutoff,\'evaluation\')\n    selected=np.flatnonzero(sample.split.to_numpy()==\'evaluation\')\n    search_missed=0\n    for i in selected:\n        retrieved={records[int(r)][0] for r in ids[i,:20] if r>=0}\n        search_missed+=len(truth[sample.iloc[i].entity_id]-retrieved)\n    evaluation[\'missed_by_search\']=search_missed\n    evaluation[\'retrieved_but_rejected\']=evaluation[\'missed_true_links\']-search_missed\n    result={\'version\':VERSION,\'model_id\':MODEL_ID,\'revision\':REVISION,\'candidate_k\':20,\n            \'ann_nprobe\':64,\'ann_overfetch\':200,\'threshold\':cutoff,\'tuning\':best,\n            \'evaluation\':evaluation,\'features\':MATCH_FEATURES,\n            \'scope\':\'Full training S2/S3; 3000 S1 train, 500 tune, 500 evaluation; no test predictions.\',\n            \'versions\':{p:version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'faiss-cpu\',\'torch\',\'transformers\']}}\n    atomic_json(work/\'experiment_results.json\',result)\n    pd.DataFrame(tuning).to_csv(work/\'threshold_tuning.csv\',index=False)\n    with (work/\'matcher.pkl\').open(\'wb\') as f: pickle.dump({\'model\':model,\'threshold\':cutoff,\'features\':MATCH_FEATURES},f)\n    # Save diagnostics, clearly labeled TRAINING evaluation, not a portal submission.\n    diag=pd.DataFrame({\'source1_id\':sample.entity_id.to_numpy()[owners],\'candidate_id\':candidate_ids,\n                       \'split\':splits,\'label\':y,\'score\':prob})\n    diag[diag.split!=\'train\'].to_csv(work/\'heldout_pair_scores.tsv\',sep=\'\\t\',index=False)\n    log(json.dumps(result,indent=2))\n    return result\n', 'submit_pipeline.py': '"""Run the evaluated E5 + matcher pipeline on the complete challenge test set.\n\nNo fitting or threshold selection occurs here. Candidate exports contain exactly\nall final candidate pairs passed to the saved matcher.\n"""\nimport argparse, csv, gc, hashlib, json, os, pickle, shutil, subprocess, sys, time, zipfile\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom threadpoolctl import threadpool_limits\nimport embedding_core as core\n\nVERSION=\'embedding-deadline-v2\'\nPREDICT_CHUNK=2000\n\ndef sha256(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef locate_model(explicit=None):\n    if explicit:\n        candidates=[Path(explicit)]\n    else:\n        candidates=[]\n        local=Path(\'/kaggle/working/full_pool_embedding_v1\')\n        if (local/\'matcher.pkl\').exists():candidates.append(local)\n        for path in Path(\'/kaggle/input\').rglob(\'experiment_results.json\'):\n            parent=path.parent\n            if (parent/\'matcher.pkl\').exists():candidates.append(parent)\n    valid=[]\n    for path in sorted(set(candidates)):\n        result=path/\'experiment_results.json\'\n        if result.exists() and (path/\'matcher.pkl\').exists():\n            try:meta=json.loads(result.read_text())\n            except (ValueError,OSError):continue\n            if meta.get(\'version\')==\'full-pool-e5-v1\' and meta.get(\'model_id\')==core.MODEL_ID:valid.append(path)\n    if len(valid)!=1:\n        raise RuntimeError(\'Attach the completed full-pool experiment notebook OUTPUT as an input, \'\n          \'then set MODEL_DIR to the folder containing matcher.pkl and experiment_results.json. \'\n          f\'Found {len(valid)} matching folders: {[str(p) for p in valid]}\')\n    return valid[0]\n\ndef load_model(model_dir):\n    import sklearn\n    meta=json.loads((model_dir/\'experiment_results.json\').read_text())\n    assert meta[\'revision\']==core.REVISION and meta[\'features\']==core.MATCH_FEATURES\n    assert meta[\'candidate_k\']==20 and (meta[\'ann_nprobe\'],meta[\'ann_overfetch\']) in [(64,200),(128,400)]\n    assert sklearn.__version__==meta[\'versions\'][\'scikit-learn\'], (\n        \'Install scikit-learn==\'+meta[\'versions\'][\'scikit-learn\']+\' before loading this model.\')\n    # This must be YOUR trusted matcher produced by the previous notebook.\n    with (model_dir/\'matcher.pkl\').open(\'rb\') as f:bundle=pickle.load(f)\n    assert bundle[\'features\']==core.MATCH_FEATURES\n    assert abs(float(bundle[\'threshold\'])-float(meta[\'threshold\']))<1e-10\n    assert bundle[\'model\'].n_features_in_==len(core.MATCH_FEATURES)\n    assert list(bundle[\'model\'].classes_)==[0,1]\n    bundle[\'ann_nprobe\']=meta[\'ann_nprobe\'];bundle[\'ann_overfetch\']=meta[\'ann_overfetch\']\n    return bundle,meta\n\ndef initialize(data,work,model_dir):\n    data=Path(data);work=Path(work);model_dir=Path(model_dir)\n    work.mkdir(parents=True,exist_ok=True)\n    files=[data/f\'test/test_source{i}.tsv\' for i in (1,2,3)]\n    validator=data.parent/\'utils/validate_submission.py\'\n    for path in files+[validator]:\n        if not path.exists():raise FileNotFoundError(path)\n    bundle,meta=load_model(model_dir)\n    core.log(\'Checking input fingerprints and saved model...\')\n    config={\'version\':VERSION,\'inputs\':{p.name:sha256(p) for p in files},\n      \'matcher_sha256\':sha256(model_dir/\'matcher.pkl\'),\'model_id\':core.MODEL_ID,\n      \'revision\':core.REVISION,\'max_length\':256,\'dim\':core.DIM,\n      \'candidate_k\':20,\'nprobe\':meta[\'ann_nprobe\'],\'overfetch\':meta[\'ann_overfetch\'],\'threshold\':float(bundle[\'threshold\']),\n      \'prediction_chunk\':PREDICT_CHUNK,\n      \'core_sha256\':sha256(Path(core.__file__)),\'runner_sha256\':sha256(Path(__file__))}\n    cp=work/\'submission_config.json\'\n    if cp.exists():assert json.loads(cp.read_text())==config,\'Inputs, model or code changed. Use a NEW WORK directory.\'\n    else:core.atomic_json(cp,config)\n    artifact=work/\'model\';artifact.mkdir(exist_ok=True)\n    for name in [\'matcher.pkl\',\'experiment_results.json\']:\n        dest=artifact/name\n        if dest.resolve()!=(model_dir/name).resolve():shutil.copy2(model_dir/name,dest)\n    core.log(f"Saved matcher cutoff: {bundle[\'threshold\']:.3f}; 20 candidates; {meta[\'ann_nprobe\']} cells / {meta[\'ann_overfetch\']} intermediate neighbors")\n    return bundle,meta,config\n\nclass TestSearcher:\n    def __init__(self,work,nprobe=64,overfetch=200):\n        import faiss\n        self.work=work;self.faiss=faiss;self.overfetch=overfetch\n        self.total=json.loads((work/\'encoding_complete.json\').read_text())[\'total\']\n        self.vectors=np.memmap(work/\'vectors.f16\',mode=\'r\',dtype=np.float16,shape=(self.total,core.DIM))\n        self.db=sqlite3.connect(f\'file:{(work/"records.sqlite").resolve()}?mode=ro\',uri=True)\n        self.db.execute(\'PRAGMA cache_size=-65536\')\n        self.indexes={}\n        for item in json.loads((work/\'index_catalog.json\').read_text()):\n            index=faiss.read_index(str(work/item[\'file\']))\n            assert index.ntotal==item[\'count\']\n            if hasattr(index,\'nprobe\'):index.nprobe=min(nprobe,index.nlist)\n            self.indexes[item[\'country\']]=index\n    def search(self,frame,q):\n        """Same approximate search and float16-vector cosine reranking as evaluation."""\n        ids=np.full((len(frame),20),-1,np.int64)\n        scores=np.full((len(frame),20),-1,np.float32)\n        countries=frame.country.map(core.norm).to_numpy()\n        for country in np.unique(countries):\n            qi=np.flatnonzero(countries==country)\n            index=self.indexes.get(country)\n            if index is None:continue\n            for start in range(0,len(qi),64):\n                indices=qi[start:start+64]\n                _,approx=index.search(np.ascontiguousarray(q[indices],dtype=np.float32),self.overfetch)\n                # Batch gathers reduce Python/disk overhead; re-ranking remains cosine.\n                valid=approx>=0\n                safe=np.maximum(approx,0)\n                x=np.asarray(self.vectors[safe],dtype=np.float32)\n                x/=np.maximum(np.linalg.norm(x,axis=2,keepdims=True),1e-12)\n                cos=np.einsum(\'bkd,bd->bk\',x,q[indices],optimize=False)\n                cos[~valid]=-np.inf\n                for local,i in enumerate(indices):\n                    positions=np.flatnonzero(valid[local])\n                    if not len(positions):continue\n                    order=np.lexsort((approx[local,positions],-cos[local,positions]))\n                    chosen=positions[order[:20]]\n                    ids[i,:len(chosen)]=approx[local,chosen]\n                    scores[i,:len(chosen)]=cos[local,chosen]\n                del x\n        return ids,scores\n    def fetch(self,ids):\n        valid=np.unique(ids[ids>=0]);rows={}\n        for start in range(0,len(valid),800):\n            batch=[int(x) for x in valid[start:start+800]]\n            sql=\'SELECT * FROM records WHERE rid IN (\'+\',\'.join(\'?\'*len(batch))+\')\'\n            for row in self.db.execute(sql,batch):rows[row[0]]=tuple(row[1:])\n        assert len(rows)==len(valid),\'Candidate IDs missing from test record store.\'\n        return rows\n    def close(self):\n        self.db.close();self.indexes.clear();del self.vectors\n\n# sqlite is separate from core to keep this module directly runnable.\nimport sqlite3\n\ndef score_candidates(frame,ids,scores,records,bundle):\n    candidates=[[] for _ in range(len(frame))];accepted=[[] for _ in range(len(frame))]\n    features=[];owners=[];pair_ids=[]\n    for i,left in enumerate(frame[core.COLS].itertuples(index=False,name=None)):\n        seen=set()\n        for rank,rid in enumerate(ids[i]):\n            if rid<0:continue\n            right=records[int(rid)]\n            assert core.norm(left[3])==core.norm(right[3]),\'Country mismatch in retrieval.\'\n            assert right[0].startswith((\'S2-\',\'S3-\'))\n            assert right[0] not in seen,\'Duplicate candidate entity ID in test input/retrieval.\'\n            seen.add(right[0]);candidates[i].append(right[0])\n            features.append(core.features(left,right)+[float(scores[i,rank])])\n            owners.append(i);pair_ids.append(right[0])\n    if features:\n        x=np.asarray(features,np.float32)\n        assert np.isfinite(x).all()\n        prob=bundle[\'model\'].predict_proba(x)[:,1]\n        assert np.isfinite(prob).all()\n        for owner,cid,score in zip(owners,pair_ids,prob):\n            if score>=bundle[\'threshold\']:accepted[owner].append(cid)\n    assert sum(map(len,candidates))==len(features)\n    for cand,match in zip(candidates,accepted):\n        assert len(cand)<=20 and set(match)<=set(cand)\n    return candidates,accepted\n\ndef write_shard(path,frame,candidates,accepted,start):\n    staging=path.with_name(path.name+\'.tmp\');staging.mkdir(parents=True,exist_ok=True)\n    for name,values,col in [(\'candidate_pairs.tsv\',candidates,\'candidate_entity_ids\'),\n                            (\'matching_results.tsv\',accepted,\'matched_entity_ids\')]:\n        with (staging/name).open(\'w\',encoding=\'utf-8\',newline=\'\') as f:\n            writer=csv.writer(f,delimiter=\'\\t\',lineterminator=\'\\n\')\n            writer.writerow([\'source1_entity_id\',col])\n            for sid,links in zip(frame.entity_id,values):writer.writerow([sid,\',\'.join(links)])\n            f.flush();os.fsync(f.fileno())\n    stats={\'start\':start,\'rows\':len(frame),\'first\':str(frame.entity_id.iloc[0]),\'last\':str(frame.entity_id.iloc[-1]),\n       \'candidate_pairs\':sum(map(len,candidates)),\'predicted_links\':sum(map(len,accepted)),\n       \'empty_candidates\':sum(not x for x in candidates),\'empty_matches\':sum(not x for x in accepted),\n       \'candidate_histogram\':np.bincount([len(x) for x in candidates],minlength=21).tolist()}\n    core.atomic_json(staging/\'stats.json\',stats)\n    os.replace(staging,path)\n    return stats\n\ndef predict(data,work,bundle,encoder):\n    shards=work/\'shards\';shards.mkdir(exist_ok=True)\n    searcher=TestSearcher(work,bundle.get("ann_nprobe",64),bundle.get("ann_overfetch",200))\n    counts={};start=0;started=time.time();new=0\n    total_s1=sum(len(c) for c in core.read_chunks(data/\'test/test_source1.tsv\',[\'entity_id\']))\n    core.log(f\'Predicting all {total_s1:,} test Source 1 businesses; output checkpoint every {PREDICT_CHUNK:,}.\')\n    try:\n        with threadpool_limits(limits=4):\n            searcher.faiss.omp_set_num_threads(min(4,os.cpu_count() or 2))\n            for frame in pd.read_csv(data/\'test/test_source1.tsv\',sep=\'\\t\',dtype=str,\n                                      keep_default_na=False,usecols=core.COLS,chunksize=PREDICT_CHUNK):\n                frame=frame[core.COLS].reset_index(drop=True)\n                path=shards/f\'{start:010d}\'\n                if path.exists():\n                    stats=json.loads((path/\'stats.json\').read_text())\n                    assert stats[\'start\']==start and stats[\'rows\']==len(frame)\n                    assert stats[\'first\']==frame.entity_id.iloc[0] and stats[\'last\']==frame.entity_id.iloc[-1]\n                    assert all((path/n).exists() for n in [\'candidate_pairs.tsv\',\'matching_results.tsv\'])\n                else:\n                    if shutil.disk_usage(work).free<1.5*2**30:raise RuntimeError(\'Low disk. Completed shards saved; free space before rerunning.\')\n                    q=encoder.encode(frame)\n                    ids,scores=searcher.search(frame,q)\n                    records=searcher.fetch(ids)\n                    cand,accepted=score_candidates(frame,ids,scores,records,bundle)\n                    stats=write_shard(path,frame,cand,accepted,start)\n                    new+=len(frame)\n                    del records,q,ids,scores,cand,accepted\n                    if new==len(frame) or (start+len(frame))%10000==0:\n                        rate=new/max(time.time()-started,1)\n                        eta=(total_s1-start-len(frame))/max(rate,1)/60\n                        core.log(f\'{start+len(frame):,}/{total_s1:,} businesses; {rate:.1f}/sec; remaining prediction ~{eta:.1f} min\')\n                for k in [\'rows\',\'candidate_pairs\',\'predicted_links\',\'empty_candidates\',\'empty_matches\']:\n                    counts[k]=counts.get(k,0)+stats[k]\n                start+=len(frame)\n    finally:searcher.close();gc.collect()\n    assert start==total_s1\n    core.atomic_json(work/\'prediction_complete.json\',counts)\n    core.log(\'All test businesses predicted: \'+json.dumps(counts))\n    return counts\n\ndef merge_outputs(data,work):\n    counts=json.loads((work/\'prediction_complete.json\').read_text())\n    out=work/\'output\';out.mkdir(exist_ok=True)\n    paths=sorted(p for p in (work/\'shards\').iterdir() if p.is_dir() and p.name.isdigit())\n    expected=0;hist=np.zeros(21,np.int64)\n    for p in paths:\n        s=json.loads((p/\'stats.json\').read_text())\n        assert s[\'start\']==expected\n        expected+=s[\'rows\'];hist+=s[\'candidate_histogram\']\n    assert expected==counts[\'rows\']\n    for filename,second in [(\'matching_results.tsv\',\'matched_entity_ids\'),(\'candidate_pairs.tsv\',\'candidate_entity_ids\')]:\n        path=out/filename;tmp=Path(str(path)+\'.tmp\');rows=0\n        with tmp.open(\'wb\') as f:\n            header=f\'source1_entity_id\\t{second}\\n\'.encode()\n            f.write(header)\n            for p in paths:\n                with (p/filename).open(\'rb\') as src:\n                    assert src.readline()==header\n                    for line in src:\n                        f.write(line);rows+=1\n        assert rows==expected\n        os.replace(tmp,path)\n    counts[\'mean_candidates\']=counts[\'candidate_pairs\']/counts[\'rows\']\n    counts[\'candidate_histogram\']=hist.tolist()\n    core.atomic_json(out/\'submission_summary.json\',counts)\n    core.log(\'Merged submission files: \'+json.dumps(counts))\n    return out\n\ndef validate(data,work):\n    validator=data.parent/\'utils/validate_submission.py\'\n    out=work/\'output\'\n    command=[sys.executable,str(validator),\'--matching\',str(out/\'matching_results.tsv\'),\n             \'--candidate\',str(out/\'candidate_pairs.tsv\'),\'--test-dir\',str(data/\'test\'),\'--check-ids\']\n    core.log(\'Running official validator, including test ID existence checks...\')\n    with (out/\'validation.log\').open(\'w\') as log:\n        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)\n        for line in proc.stdout:\n            print(line,end=\'\',flush=True);log.write(line);log.flush()\n        code=proc.wait()\n    if code:raise RuntimeError(f\'Official validator failed (exit {code}). Do not submit. See validation.log.\')\n    core.atomic_json(out/\'validated.json\',{\'matching_sha256\':sha256(out/\'matching_results.tsv\'),\n               \'candidate_sha256\':sha256(out/\'candidate_pairs.tsv\'),\'check_ids\':True})\n    core.log(\'VALIDATOR PASSED. Portal upload: \'+str(out/\'matching_results.tsv\'))\n\ndef package(work,meta):\n    from importlib.metadata import version\n    out=work/\'output\';assert (out/\'validated.json\').exists()\n    validation=json.loads((out/\'validated.json\').read_text())\n    assert validation[\'matching_sha256\']==sha256(out/\'matching_results.tsv\')\n    assert validation[\'candidate_sha256\']==sha256(out/\'candidate_pairs.tsv\')\n    (work/\'requirements.txt\').write_text(\'\\n\'.join(f\'{p}=={version(p)}\' for p in\n             [\'torch\',\'transformers\',\'numpy\',\'pandas\',\'scikit-learn\',\'faiss-cpu\',\'safetensors\',\'sentencepiece\'])+\'\\n\')\n    (work/\'README.md\').write_text(\'\'\'# Embedding submission v2\nUpload `output/matching_results.tsv` to the leaderboard portal.\nRetain `output/candidate_pairs.tsv`, the model, code, and methodology for final review.\nThe candidate file contains exactly every final pair evaluated by the classifier.\n\n## Reproduce predictions\nInstall requirements in a compatible GPU environment. Public E5 weights are pinned by revision.\nRun: `python submit_pipeline.py --data /path/to/student_resource/dataset --model-dir ./model --work ./rerun --stage all`\nThe input dataset and official validator must be provided separately. No business lookups occur.\nAll test S1 entities receive one row, including empty predictions. Country matching is enforced.\n\n## Reproduce training\nRun `training_experiment.ipynb` on the supplied training dataset to recreate the sampled\ntraining experiment, matcher and cutoff. Its dataset and working paths are configurable.\nThe trained matcher in `model/matcher.pkl` comes from that experiment, not test labels.\nThe upgrade training notebook requires the v1 training caches as a read-only input.\nOnly load your own trusted pickle models.\n\n## Checkpoints\nThis archive excludes large vector and index caches. The complete saved Kaggle output\ncontains those, plus completed prediction shards. Rerunning against an unchanged working\nfolder resumes completed work. A discarded unsaved session cannot be resumed.\n\n## Final package\nThis archive preserves submission evidence and runnable code. Before final challenge\nclosure, transfer methodology into the organizer\'s Documentation_template.md and check\nany organizer-specific archive naming or additional requirements.\n\'\'\')\n    (work/\'methodology.md\').write_text(f\'\'\'# Methodology — embedding submission v2\nNames and addresses are NFKC/whitespace normalized, preserving Unicode. E5 encodes\n`query: business name: NAME; address: ADDRESS` with attention-masked mean pooling and\nL2 normalization, maximum 256 tokens. Model: {core.MODEL_ID}; revision: {core.REVISION}.\nPretrained weights are fixed, not fine-tuned. Business data comes only from the challenge.\n\nTest S2/S3 vectors form independent country-specific IVF-PQ indexes (up to 2048 cells,\n48 subquantizers, 8 bits each). Quantizers use up to 110000 unlabeled target vectors\nper country, seed 2026. Search probes {meta[\'ann_nprobe\']} cells, retrieves up to {meta[\'ann_overfetch\']} approximate candidates,\nthen reranks by cosine against stored float16 vectors and keeps at most 20.\nOnly these final 20 are passed to the matcher and written to candidate_pairs.tsv.\nIntermediate retrieval counts and code are disclosed here for auditability.\n\nThe HistGradientBoostingClassifier uses the 18 lexical features defined in embedding_core.py\nand embedding cosine similarity. It was trained on 30000 sampled S1 businesses (up to 600000 pairs),\nwith 2000 tuning and 2000 fresh evaluation businesses. Threshold: {meta[\'threshold\']}.\nTraining targets include the entire provided train S2/S3 pool. No truth matches are injected.\nLocal evaluation macro F0.5: {meta[\'evaluation\'][\'macro_F05\']:.6f}; this is not a test leaderboard score.\nTest predictions use the saved model and cutoff without retraining or threshold changes.\n\nSource 1 is processed in batches and files are assembled in input order. The official\nvalidator runs with --check-ids. Empty match lists are retained. France is present in test\nbut absent from labeled training; country-specific French accuracy has not been validated.\nThe country index code derives countries dynamically, including France.\n\nSee output/submission_summary.json for actual candidate counts, and requirements.txt for versions.\n\'\'\')\n    # A small ZIP is easy to download even if the browser previews TSV files.\n    with zipfile.ZipFile(out/\'matching_results_download.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        z.write(out/\'matching_results.tsv\',\'matching_results.tsv\')\n    archive=work.parent/\'embedding_submission_v2_backup.zip\'\n    members=[\'README.md\',\'methodology.md\',\'requirements.txt\',\'submission_config.json\',\n      \'embedding_core.py\',\'submit_pipeline.py\',\'training_experiment.ipynb\',\n      \'model/matcher.pkl\',\'model/experiment_results.json\',\'output/matching_results.tsv\',\n      \'output/candidate_pairs.tsv\',\'output/submission_summary.json\',\'output/validation.log\',\'output/validated.json\']\n    for name in members:\n        assert (work/name).exists(),f\'Missing backup member: {name}\'\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for name in members:z.write(work/name,name)\n    core.log(\'Downloadable matches ZIP: \'+str(out/\'matching_results_download.zip\'))\n    core.log(\'Retain backup: \'+str(archive))\n    return archive\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',type=Path,required=True)\n    parser.add_argument(\'--work\',type=Path,required=True)\n    parser.add_argument(\'--model-dir\',type=Path,required=True)\n    parser.add_argument(\'--stage\',choices=[\'all\',\'encode\',\'index\',\'predict\',\'finalize\'],default=\'all\')\n    parser.add_argument(\'--batch-size\',type=int,default=64)\n    args=parser.parse_args();data,work=args.data,args.work\n    bundle,meta,config=initialize(data,work,args.model_dir)\n    if args.stage in (\'all\',\'encode\'):\n        encoder=core.Encoder(args.batch_size)\n        core.encode_pool(data,work,encoder,split=\'test\')\n        del encoder;gc.collect()\n        import torch;torch.cuda.empty_cache()\n    if args.stage in (\'all\',\'index\'):core.build_indexes(work)\n    if args.stage in (\'all\',\'predict\'):\n        encoder=core.Encoder(args.batch_size)\n        predict(data,work,bundle,encoder)\n        del encoder;gc.collect()\n        import torch;torch.cuda.empty_cache()\n    if args.stage in (\'all\',\'finalize\'):\n        merge_outputs(data,work);validate(data,work);package(work,meta)\n\nif __name__==\'__main__\':main()\n', 'upgrade_train.py': '"""Reuse read-only training embeddings; train 30k, tune 2k, evaluate 2k fresh S1 entities."""\nimport os,sys,json,pickle,hashlib,time,gc,argparse\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nfrom zoneinfo import ZoneInfo\nimport numpy as np,pandas as pd\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom threadpoolctl import threadpool_limits\nimport embedding_core as c\nimport submit_pipeline as s\n\nSEED=20260927\nTRAIN_N=30000\nTUNE_N=2000\nEVAL_N=2000\nPROFILES={\'standard\':(64,200),\'strong\':(128,400)}\n\ndef resolve_cache(explicit=None):\n    if explicit:candidates=[Path(explicit)]\n    else:\n        candidates=[x.parent for x in Path(\'/kaggle/input\').rglob(\'encoding_complete.json\')]\n        local=Path(\'/kaggle/working/full_pool_embedding_v1\')\n        if local.exists():candidates.append(local)\n    valid=[]\n    for p in sorted(set(candidates)):\n        required=[\'vectors.f16\',\'records.sqlite\',\'index_catalog.json\',\'encoding_complete.json\',\n                  \'sample.tsv\',\'matcher.pkl\',\'experiment_results.json\',\'configuration.json\']\n        if not all((p/n).exists() for n in required):continue\n        meta=json.loads((p/\'experiment_results.json\').read_text())\n        if meta.get(\'version\')!=\'full-pool-e5-v1\':continue\n        assert meta[\'model_id\']==c.MODEL_ID and meta[\'revision\']==c.REVISION\n        count=json.loads((p/\'encoding_complete.json\').read_text())[\'total\']\n        assert (p/\'vectors.f16\').stat().st_size==count*c.DIM*2\n        catalog=json.loads((p/\'index_catalog.json\').read_text())\n        assert sum(x[\'count\'] for x in catalog)==count\n        assert all((p/x[\'file\']).exists() for x in catalog)\n        valid.append(p)\n    if len(valid)!=1:\n        raise RuntimeError(\'Attach the FULL saved v1 notebook output as input, not only the small results ZIP. \'\n          \'Set CACHE_DIR to its full_pool_embedding_v1 folder if more than one is found. Found: \'+str(valid))\n    return valid[0]\n\ndef setup(data,cache,work):\n    work.mkdir(parents=True,exist_ok=True)\n    old=json.loads((cache/\'configuration.json\').read_text())\n    assert old[\'revision\']==c.REVISION and old[\'dim\']==c.DIM and old[\'max_length\']==256\n    for path,size in old[\'files\']:\n        actual=data/\'train\'/Path(path).name\n        assert actual.exists() and actual.stat().st_size==size,\'Training dataset does not match cache.\'\n    config={\'version\':\'deadline-upgrade-v2\',\'seed\':SEED,\'train_n\':TRAIN_N,\'tune_n\':TUNE_N,\'evaluation_n\':EVAL_N,\n      \'old_matcher_sha256\':s.sha256(cache/\'matcher.pkl\'),\'old_sample_sha256\':s.sha256(cache/\'sample.tsv\'),\n      \'cache_manifest_sha256\':s.sha256(cache/\'encoding_complete.json\'),\n      \'profiles\':PROFILES,\'features\':c.MATCH_FEATURES,\n      \'core_sha256\':s.sha256(Path(c.__file__)),\'runner_sha256\':s.sha256(Path(s.__file__)),\n      \'upgrade_sha256\':s.sha256(Path(__file__))}\n    config=json.loads(json.dumps(config))\n    path=work/\'upgrade_config.json\'\n    if path.exists():assert json.loads(path.read_text())==config,\'Configuration changed. Use a new UPGRADE_WORK.\'\n    else:c.atomic_json(path,config)\n\n\ndef sample_data(data,cache,work):\n    path=work/\'sample.pkl\'\n    if path.exists():\n        with path.open(\'rb\') as f:return pickle.load(f)\n    exclude=set(pd.read_csv(cache/\'sample.tsv\',sep=\'\\t\',dtype=str,keep_default_na=False).entity_id)\n    rng=np.random.default_rng(SEED);sample=pd.DataFrame();n=TRAIN_N+TUNE_N+EVAL_N\n    for chunk in c.read_chunks(data/\'train/train_source1.tsv\'):\n        chunk=chunk.loc[~chunk.entity_id.isin(exclude),c.COLS].copy()\n        chunk[\'_priority\']=rng.random(len(chunk))\n        sample=pd.concat([sample,chunk],ignore_index=True).nsmallest(n,\'_priority\')\n    sample=sample.drop(columns=\'_priority\').reset_index(drop=True)\n    assert len(sample)==n and sample.entity_id.is_unique\n    assert not set(sample.entity_id)&exclude\n    ids=set(sample.entity_id);truth={}\n    for chunk in c.read_chunks(data/\'train/train_ground_truth.tsv\',[\'source1_entity_id\',\'matched_entity_ids\']):\n        for r in chunk[chunk.source1_entity_id.isin(ids)].itertuples(index=False):\n            truth[r.source1_entity_id]=set(x for x in r.matched_entity_ids.split(\',\') if x)\n    assert set(truth)==ids\n    strata=sample.country+\'|\'+sample.entity_id.map(lambda x:str(not truth[x]))\n    _,held=train_test_split(np.arange(n),test_size=TUNE_N+EVAL_N,random_state=SEED,stratify=strata)\n    tune,evaluation=train_test_split(held,test_size=EVAL_N,random_state=SEED+1,stratify=strata.iloc[held])\n    sample[\'split\']=\'train\';sample.loc[tune,\'split\']=\'tune\';sample.loc[evaluation,\'split\']=\'evaluation\'\n    sample.to_csv(work/\'sample.tsv\',sep=\'\\t\',index=False)\n    with Path(str(path)+\'.tmp\').open(\'wb\') as f:pickle.dump((sample,truth),f)\n    os.replace(str(path)+\'.tmp\',path)\n    return sample,truth\n\n\ndef queries(sample,work,encoder):\n    path=work/\'queries.npy\'\n    if path.exists():return np.load(path)\n    out=[]\n    for start in range(0,len(sample),1000):\n        out.append(encoder.encode(sample.iloc[start:start+1000]))\n        c.log(f\'Query embeddings: {min(start+1000,len(sample)):,}/{len(sample):,}\')\n    q=np.concatenate(out)\n    with Path(str(path)+\'.tmp\').open(\'wb\') as f:np.save(f,q)\n    os.replace(str(path)+\'.tmp\',path)\n    return q\n\n\ndef pair_arrays(frame,truth,owners_global,ids,scores,records):\n    xs=[];ys=[];owners=[]\n    for local,left in enumerate(frame[c.COLS].itertuples(index=False,name=None)):\n        for rank,rid in enumerate(ids[local]):\n            if rid<0:continue\n            right=records[int(rid)]\n            assert c.norm(left[3])==c.norm(right[3])\n            xs.append(c.features(left,right)+[float(scores[local,rank])])\n            ys.append(int(right[0] in truth[left[0]]));owners.append(int(owners_global[local]))\n    return dict(x=np.asarray(xs,np.float32).reshape(-1,len(c.MATCH_FEATURES)),\n                y=np.asarray(ys,np.uint8),owners=np.asarray(owners,np.int32))\n\n\ndef prepare_pairs(cache,work,sample,truth,q,profile,indices):\n    npb,retr=PROFILES[profile]\n    folder=work/f\'pairs_{profile}\';folder.mkdir(exist_ok=True)\n    search=s.TestSearcher(cache,nprobe=npb,overfetch=retr);search.faiss.omp_set_num_threads(4)\n    pieces=[];started=time.time()\n    try:\n        with threadpool_limits(limits=4):\n            for start in range(0,len(indices),500):\n                selected=indices[start:start+500];path=folder/f\'{start:07d}.npz\'\n                if path.exists():\n                    with np.load(path) as z:\n                        assert np.array_equal(z[\'query_indices\'],selected)\n                        piece={k:z[k].copy() for k in [\'x\',\'y\',\'owners\']}\n                else:\n                    frame=sample.iloc[selected].reset_index(drop=True)\n                    ids,scores=search.search(frame,q[selected]);records=search.fetch(ids)\n                    piece=pair_arrays(frame,truth,selected,ids,scores,records)\n                    with Path(str(path)+\'.tmp\').open(\'wb\') as f:np.savez(f,query_indices=selected,**piece)\n                    os.replace(str(path)+\'.tmp\',path)\n                    del records,ids,scores\n                pieces.append(piece)\n                if start==0 or (start+len(selected))%2500==0:\n                    c.log(f\'{profile}: {start+len(selected):,}/{len(indices):,} businesses; {(time.time()-started)/60:.1f} min\')\n    finally:search.close()\n    return {k:np.concatenate([piece[k] for piece in pieces]) for k in [\'x\',\'y\',\'owners\']}\n\n\ndef metrics(prob,pairs,selected,truth_counts,threshold):\n    n=len(truth_counts);accepted=prob>=threshold\n    predicted=np.bincount(pairs[\'owners\'][accepted],minlength=n)\n    tp=np.bincount(pairs[\'owners\'][accepted & (pairs[\'y\']==1)],minlength=n)\n    fp=predicted-tp;fn=truth_counts-tp\n    denom=1.25*tp+fp+.25*fn\n    score=np.divide(1.25*tp,denom,out=np.zeros(n,dtype=float),where=denom!=0)\n    score[(truth_counts==0)&(predicted==0)]=1\n    singleton=truth_counts[selected]==0\n    return dict(macro_F05=float(score[selected].mean()),correct_accepted=int(tp[selected].sum()),\n       incorrect_accepted=int(fp[selected].sum()),missed_true_links=int(fn[selected].sum()),\n       singletons_correct=int(((predicted[selected]==0)&singleton).sum()),singletons=int(singleton.sum()))\n\n\ndef tune(prob,pairs,selected,counts):\n    trials=[dict(threshold=float(t),**metrics(prob,pairs,selected,counts,t)) for t in np.linspace(.05,.95,91)]\n    return max(trials,key=lambda x:(x[\'macro_F05\'],x[\'threshold\'])),trials\n\n\ndef retrieval(pairs,selected,counts):\n    hits=np.bincount(pairs[\'owners\'],weights=pairs[\'y\'],minlength=len(counts))\n    sizes=np.bincount(pairs[\'owners\'],minlength=len(counts))\n    found=int(hits[selected].sum());total=int(counts[selected].sum())\n    return dict(found=found,true_links=total,recall=found/total if total else 1,\n                mean_candidates=float(sizes[selected].mean()),max_candidates=int(sizes[selected].max()))\n\n\ndef benchmark(cache,sample,encoder,bundle,profile):\n    # Raw text -> embedding -> search -> lexical features -> model score, not search alone.\n    npb,over=PROFILES[profile];search=s.TestSearcher(cache,npb,over)\n    frame=sample.sample(n=min(1000,len(sample)),random_state=42).reset_index(drop=True)\n    elapsed=[]\n    try:\n        with threadpool_limits(limits=4):\n            search.faiss.omp_set_num_threads(4)\n            for start in range(0,len(frame),250):\n                part=frame.iloc[start:start+250].reset_index(drop=True)\n                t=time.time();q=encoder.encode(part);ids,score=search.search(part,q)\n                records=search.fetch(ids);s.score_candidates(part,ids,score,records,bundle)\n                elapsed.append(time.time()-t)\n    finally:search.close()\n    return len(frame)/sum(elapsed)\n\n\ndef run(data,cache,work,target_finish,batch_size=64):\n    data=Path(data);cache=Path(cache);work=Path(work)\n    setup(data,cache,work)\n    old_bundle,old_meta=s.load_model(cache)\n    sample,truth=sample_data(data,cache,work)\n    c.log(\'Fresh sample: \'+str(sample.split.value_counts().to_dict()))\n    encoder=c.Encoder(batch_size)\n    q=queries(sample,work,encoder)\n    strong=prepare_pairs(cache,work,sample,truth,q,\'strong\',np.arange(len(sample)))\n    held=np.flatnonzero(sample.split.to_numpy()!=\'train\')\n    standard=prepare_pairs(cache,work,sample,truth,q,\'standard\',held)\n    train_mask=sample.split.to_numpy()[strong[\'owners\']]==\'train\'\n    saved=work/\'new_classifier.pkl\'\n    if saved.exists():\n        with saved.open(\'rb\') as f:model=pickle.load(f)\n    else:\n        c.log(f\'Training 30k-business matcher: {train_mask.sum():,} pairs; {strong["y"][train_mask].sum():,} positives\')\n        model=HistGradientBoostingClassifier(learning_rate=.08,max_iter=250,max_leaf_nodes=31,\n                    l2_regularization=1.,early_stopping=False,random_state=2026)\n        with threadpool_limits(limits=4):model.fit(strong[\'x\'][train_mask],strong[\'y\'][train_mask])\n        with Path(str(saved)+\'.tmp\').open(\'wb\') as f:pickle.dump(model,f)\n        os.replace(str(saved)+\'.tmp\',saved)\n    strong={k:a[~train_mask] for k,a in strong.items()}\n    pairs={\'standard\':standard,\'strong\':strong}\n    counts=np.array([len(truth[x]) for x in sample.entity_id])\n    ti=np.flatnonzero(sample.split.to_numpy()==\'tune\');ei=np.flatnonzero(sample.split.to_numpy()==\'evaluation\')\n    scores={};options={};benchmark_rates={};all_trials=[]\n    total_test=sum(len(chunk) for chunk in c.read_chunks(data/\'test/test_source1.tsv\',[\'entity_id\']))\n    for name,arr in pairs.items():\n        with threadpool_limits(limits=4):scores[name]=model.predict_proba(arr[\'x\'])[:,1]\n        best,trials=tune(scores[name],arr,ti,counts);options[name]=best\n        all_trials.extend(dict(profile=name,**r) for r in trials)\n        bundle={\'model\':model,\'threshold\':best[\'threshold\']}\n        benchmark_rates[name]=benchmark(cache,sample,encoder,bundle,name)\n        c.log(f\'{name}: tuning macro F0.5={best["macro_F05"]:.5f}; prediction benchmark={benchmark_rates[name]:.1f} businesses/sec\')\n    # Profile and threshold chosen using tuning scores and time budget, before evaluation.\n    finish=datetime.fromisoformat(target_finish)\n    assert finish.tzinfo is not None,\'Use an ISO timestamp with timezone, e.g. +05:30.\'\n    remaining_hours=(finish-datetime.now(timezone.utc)).total_seconds()/3600\n    # 35% slower than warm training benchmark, plus 2h test encoding/index/export allowance.\n    estimates={name:total_test/rate/3600*1.35+2.0 for name,rate in benchmark_rates.items()}\n    feasible=[name for name in pairs if estimates[name]<=remaining_hours]\n    chosen=max(feasible or list(pairs),key=lambda name:(options[name][\'macro_F05\'],name==\'standard\'))\n    cutoff=options[chosen][\'threshold\']\n    with threadpool_limits(limits=4):old_prob=old_bundle[\'model\'].predict_proba(standard[\'x\'])[:,1]\n    baseline=metrics(old_prob,standard,ei,counts,old_bundle[\'threshold\'])\n    evaluation=metrics(scores[chosen],pairs[chosen],ei,counts,cutoff)\n    search_eval=retrieval(pairs[chosen],ei,counts)\n    evaluation[\'missed_by_search\']=search_eval[\'true_links\']-search_eval[\'found\']\n    evaluation[\'retrieved_but_rejected\']=evaluation[\'missed_true_links\']-evaluation[\'missed_by_search\']\n    # Only one fresh evaluation comparison; retain the already-submitted fallback on regression.\n    promote=bool(feasible and evaluation[\'macro_F05\']>baseline[\'macro_F05\'])\n    report={\'old_model_fresh_evaluation\':baseline,\'new_model_fresh_evaluation\':evaluation,\n       \'tuning_options\':options,\'chosen_profile\':chosen,\'threshold\':cutoff,\n       \'search_recall_evaluation\':search_eval,\'prediction_benchmark_per_second\':benchmark_rates,\n       \'estimated_test_hours_with_margin\':estimates,\'hours_until_target_finish\':remaining_hours,\n       \'target_finish\':target_finish,\'promote_to_test\':promote,\n       \'note\':\'Timing is an estimate, not a guarantee. Evaluation excludes all previous 4000 sampled S1 records.\'}\n    c.atomic_json(work/\'upgrade_report.json\',report)\n    pd.DataFrame(all_trials).to_csv(work/\'threshold_tuning.csv\',index=False)\n    by_country=[]\n    for country in sorted(sample.country.unique()):\n        selected=ei[sample.country.to_numpy()[ei]==country]\n        by_country.append({\'country\':country,\'businesses\':len(selected),\n           \'old\':metrics(old_prob,standard,selected,counts,old_bundle[\'threshold\']),\n           \'new\':metrics(scores[chosen],pairs[chosen],selected,counts,cutoff)})\n    c.atomic_json(work/\'country_evaluation.json\',by_country)\n    model_dir=work/\'model\';model_dir.mkdir(exist_ok=True)\n    bundle={\'model\':model,\'threshold\':cutoff,\'features\':c.MATCH_FEATURES,\n            \'ann_nprobe\':PROFILES[chosen][0],\'ann_overfetch\':PROFILES[chosen][1]}\n    with (model_dir/\'matcher.pkl\').open(\'wb\') as f:pickle.dump(bundle,f)\n    from importlib.metadata import version\n    meta={\'version\':\'deadline-upgrade-v2\',\'model_id\':c.MODEL_ID,\'revision\':c.REVISION,\'candidate_k\':20,\n       \'ann_nprobe\':PROFILES[chosen][0],\'ann_overfetch\':PROFILES[chosen][1],\'threshold\':cutoff,\n       \'evaluation\':evaluation,\'tuning\':options[chosen],\'features\':c.MATCH_FEATURES,\n       \'train_businesses\':TRAIN_N,\'tune_businesses\':TUNE_N,\'evaluation_businesses\':EVAL_N,\n       \'versions\':{p:version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'faiss-cpu\',\'torch\',\'transformers\']}}\n    c.atomic_json(model_dir/\'experiment_results.json\',meta)\n    print(json.dumps(report,indent=2),flush=True)\n    print(\'AUTO TEST RUN APPROVED\' if promote else \'KEEP 0.84 FALLBACK: validation did not improve or estimated runtime exceeds target.\',flush=True)\n    del encoder;gc.collect()\n    import torch;torch.cuda.empty_cache()\n    return report\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser();ap.add_argument(\'--data\',type=Path,required=True)\n    ap.add_argument(\'--cache\',type=Path,required=True);ap.add_argument(\'--work\',type=Path,required=True)\n    ap.add_argument(\'--target-finish\',required=True);ap.add_argument(\'--batch-size\',type=int,default=64)\n    args=ap.parse_args();run(args.data,args.cache,args.work,args.target_finish,args.batch_size)\n'}
for name,text in MODULES.items():
    (CODE_DIR/name).write_text(text)
probe=subprocess.run([sys.executable,'-c',
    'import upgrade_train as u; print(u.resolve_cache('+repr(CACHE_DIR or None)+'))'],
    cwd=CODE_DIR,text=True,capture_output=True)
if probe.returncode:
    print(probe.stderr)
    raise RuntimeError('Attach the FULL saved previous notebook output and check CACHE_DIR.')
CACHE_DIR=Path(probe.stdout.strip())
print('Read-only training cache:',CACHE_DIR)
print('Training target vectors will NOT be regenerated or copied.')

def run_logged(command,name):
    with (ROOT/name).open('a') as log:
        proc=subprocess.Popen(command,cwd=CODE_DIR,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        returncode=proc.wait()
    if returncode:
        raise RuntimeError('Stage stopped. See the preceding log. Completed checkpoints remain in ROOT.')


## 1. Train and compare using the saved training pool
This draws a new sample excluding **all 4,000 businesses from the old experiment**. It generates real retrieval candidates, without injecting ground-truth matches. Training and pair-feature batches are checkpointed.

The old model is scored using its original cutoff and standard search. The new model's cutoff and search profile use only the tuning businesses. Both are then compared on the same fresh evaluation businesses. This is a local comparison, not a guarantee of a leaderboard gain; France remains absent from labeled training.

The runtime benchmark includes query encoding, retrieval, feature computation, and classifier prediction. Its forecast adds 35% to benchmarked prediction time plus two hours for test encoding/indexing/export. Cold disk access and GPU variation can still make the full run slower.


In [ ]:
run_logged([sys.executable,str(CODE_DIR/'upgrade_train.py'),
    '--data',str(DATA),'--cache',str(CACHE_DIR),'--work',str(UPGRADE_WORK),
    '--target-finish',TARGET_FINISH,'--batch-size',str(BATCH_SIZE)],'upgrade_training.log')
report=json.loads((UPGRADE_WORK/'upgrade_report.json').read_text())
print(json.dumps(report,indent=2))


## 2. Conditional test submission run
If the new model improves fresh evaluation and the runtime estimate fits, this starts the full test run with the selected model, cutoff and search profile. It does not upload anything to the portal.

The 20 candidates scored for each business are all written to `candidate_pairs.tsv`. The accepted subset goes to `matching_results.tsv`. All test S1 businesses get rows, including empty predictions. Test predictions checkpoint every 2,000 businesses. Encoding also checkpoints.

If the gate says to keep the fallback, no expensive test run starts. Share the report rather than overriding the gate under deadline pressure. A better local score does not guarantee a better leaderboard score.


In [ ]:
TRAINING_NOTEBOOK='{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "# Amazon ML \\u2014 deadline upgrade, cached training + submission\\n",\n    "\\n",\n    "Keep your submitted **0.84** result as the fallback. This notebook attempts one measured improvement:\\n",\n    "- **30,000 training businesses**, 2,000 tuning businesses, and 2,000 fresh evaluation businesses.\\n",\n    "- Same fixed E5 model and 19 comparison features; more labeled examples for the matcher.\\n",\n    "- Compare standard search (64 cells / 200 intermediate neighbors) with stronger search (128 / 400), always **20 final candidates**.\\n",\n    "- Choose search and cutoff on tuning data and estimated runtime. Compare the chosen new pipeline to the saved old model on the same fresh evaluation businesses.\\n",\n    "- Start full test prediction automatically **only if evaluation improves and the estimated time fits the target**.\\n",\n    "\\n",\n    "## Inputs and settings\\n",\n    "Use a new Kaggle notebook. Attach **(1) the original challenge dataset and (2) the full saved output of the previous notebook**. The second input needs `full_pool_embedding_v1/vectors.f16`, `records.sqlite`, country `.faiss` files, `sample.tsv`, `matcher.pkl`, and metadata. The small results ZIP alone is insufficient.\\n",\n    "\\n",\n    "Enable GPU + Internet. Do not copy the large training caches to working storage: they are read directly from the attached input. Training target embeddings/indexes are reused. Test embeddings still need to be generated.\\n",\n    "\\n",\n    "The planned finish is **27 September 2026, 10:30 p.m. IST**, leaving time before the stated 11:59 p.m. deadline. Estimates include a margin but are not guarantees. Use **Save Version \\u2192 Save & Run All**. Expect the evaluation stage to finish much sooner than full submission prediction; watch its report.\\n"\n   ],\n   "id": "training-0"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "%pip install -q \\"numpy==2.0.2\\" \\"pandas==2.3.3\\" \\"scikit-learn==1.6.1\\" \\"faiss-cpu==1.11.0\\" \\"transformers==5.0.0\\" safetensors sentencepiece\\n"\n   ],\n   "outputs": [],\n   "execution_count": null,\n   "id": "training-1"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "from pathlib import Path\\n",\n    "import os,sys,json,subprocess,shutil\\n",\n    "DATA=Path(\'/kaggle/input/datasets/spoortimorabad/ml-challenge-dataset/student_resource/dataset\')\\n",\n    "# Leave blank for automatic discovery. If needed, set to the attached full_pool_embedding_v1 folder.\\n",\n    "CACHE_DIR=\'\'\\n",\n    "ROOT=Path(\'/kaggle/working/ml_deadline_upgrade_v2\')\\n",\n    "CODE_DIR=ROOT/\'code\'\\n",\n    "UPGRADE_WORK=ROOT/\'training_upgrade\'\\n",\n    "SUBMISSION_WORK=ROOT/\'submission\'\\n",\n    "TARGET_FINISH=\'2026-09-27T22:30:00+05:30\'\\n",\n    "BATCH_SIZE=64\\n",\n    "CODE_DIR.mkdir(parents=True,exist_ok=True)\\n",\n    "os.environ[\'TOKENIZERS_PARALLELISM\']=\'false\'\\n",\n    "import torch\\n",\n    "assert torch.cuda.is_available(),\'Enable a GPU in notebook settings.\'\\n",\n    "assert (DATA/\'train/train_source1.tsv\').exists(),\'Update DATA to the challenge dataset path.\'\\n",\n    "assert (DATA.parent/\'utils/validate_submission.py\').exists(),\'The original challenge utils folder is required.\'\\n",\n    "print(\'GPU:\',torch.cuda.get_device_name(0))\\n",\n    "print(\'Free working disk GiB:\',round(shutil.disk_usage(ROOT).free/2**30,2))\\n"\n   ],\n   "outputs": [],\n   "execution_count": null,\n   "id": "training-2"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "MODULES={\'embedding_core.py\': \'import os, json, time, gc, sqlite3, hashlib, shutil, pickle, re, unicodedata\\\\nfrom pathlib import Path\\\\nfrom functools import lru_cache\\\\nimport numpy as np\\\\nimport pandas as pd\\\\nfrom sklearn.ensemble import HistGradientBoostingClassifier\\\\nfrom threadpoolctl import threadpool_limits\\\\nCOLS = [\\\\\'entity_id\\\\\', \\\\\'business_name\\\\\', \\\\\'business_address\\\\\', \\\\\'country\\\\\']\\\\n\\\\nLEGAL = set(\\\\\'inc incorporated llc llp ltd limited pvt private corp corporation company co plc sarl sas eurl\\\\\'.split())\\\\n\\\\nADDR_ALIAS = {\\\\\'road\\\\\': \\\\\'rd\\\\\', \\\\\'street\\\\\': \\\\\'st\\\\\', \\\\\'avenue\\\\\': \\\\\'ave\\\\\', \\\\\'drive\\\\\': \\\\\'dr\\\\\', \\\\\'lane\\\\\': \\\\\'ln\\\\\', \\\\\'boulevard\\\\\': \\\\\'blvd\\\\\', \\\\\'apartment\\\\\': \\\\\'apt\\\\\', \\\\\'suite\\\\\': \\\\\'ste\\\\\'}\\\\n\\\\nFEATURES = [\\\\\'name_exact\\\\\', \\\\\'core_name_exact\\\\\', \\\\\'name_trigram_jaccard\\\\\', \\\\\'name_token_jaccard\\\\\', \\\\\'name_token_containment\\\\\', \\\\\'name_length_ratio\\\\\', \\\\\'address_exact\\\\\', \\\\\'address_trigram_jaccard\\\\\', \\\\\'address_token_jaccard\\\\\', \\\\\'address_token_containment\\\\\', \\\\\'address_missing\\\\\', \\\\\'number_jaccard\\\\\', \\\\\'number_disjoint\\\\\', \\\\\'unit_equal\\\\\', \\\\\'unit_conflict\\\\\', \\\\\'country_equal\\\\\', \\\\\'name_token_count\\\\\', \\\\\'candidate_name_token_count\\\\\']\\\\n\\\\ndef log(s):\\\\n    print(s, flush=True)\\\\n\\\\ndef norm(s):\\\\n    s = unicodedata.normalize(\\\\\'NFKC\\\\\', str(s)).casefold()\\\\n    return \\\\\' \\\\\'.join(re.findall(\\\\\'[^\\\\\\\\\\\\\\\\W_]+\\\\\', s))\\\\n\\\\ndef canonical_address(s):\\\\n    return \\\\\' \\\\\'.join((ADDR_ALIAS.get(t, t) for t in norm(s).split() if t not in {\\\\\'null\\\\\', \\\\\'none\\\\\'}))\\\\n\\\\ndef grams(s):\\\\n    return frozenset((s[i:i + 3] for i in range(max(0, len(s) - 2))))\\\\n\\\\ndef unit_id(s):\\\\n    m = re.search(\\\\\'\\\\\\\\\\\\\\\\b(?:unit|suite|ste|apt|apartment|flat)\\\\\\\\\\\\\\\\s*(?:no\\\\\\\\\\\\\\\\.?\\\\\\\\\\\\\\\\s*)?[:#-]?\\\\\\\\\\\\\\\\s*([\\\\\\\\\\\\\\\\w]+)\\\\\', unicodedata.normalize(\\\\\'NFKC\\\\\', s).casefold())\\\\n    return m.group(1) if m else \\\\\'\\\\\'\\\\n\\\\n@lru_cache(maxsize=30000)\\\\ndef prepared(name, address, country):\\\\n    n = norm(name)\\\\n    a = canonical_address(address)\\\\n    nt, at = (frozenset(n.split()), frozenset(a.split()))\\\\n    core = tuple(sorted((t for t in nt if t not in LEGAL)))\\\\n    nums = frozenset(re.findall(\\\\\'\\\\\\\\\\\\\\\\d+[a-z]*\\\\\', a))\\\\n    return (n, a, nt, at, core, nums, grams(n), grams(a), unit_id(address), norm(country))\\\\n\\\\ndef jaccard(a, b):\\\\n    return len(a & b) / len(a | b) if a or b else 0.0\\\\n\\\\ndef containment(a, b):\\\\n    return len(a & b) / min(len(a), len(b)) if a and b else 0.0\\\\n\\\\ndef features(left, right):\\\\n    n, a, nt, at, c, nums, ng, ag, u, country = prepared(*left[1:])\\\\n    m, b, mt, bt, d, other_nums, mg, bg, v, other_country = prepared(*right[1:])\\\\n    return [float(bool(n) and n == m), float(bool(c) and c == d), jaccard(ng, mg), jaccard(nt, mt), containment(nt, mt), min(len(n), len(m)) / max(len(n), len(m), 1), float(bool(a) and a == b), jaccard(ag, bg), jaccard(at, bt), containment(at, bt), float(not a or not b), jaccard(nums, other_nums), float(bool(nums) and bool(other_nums) and (not nums & other_nums)), float(bool(u) and bool(v) and (u == v)), float(bool(u) and bool(v) and (u != v)), float(bool(country) and country == other_country), len(nt), len(mt)]\\\\n\\\\ndef read_chunks(path, cols=COLS):\\\\n    return pd.read_csv(path, sep=\\\\\'\\\\\\\\t\\\\\', dtype=str, keep_default_na=False, usecols=cols, chunksize=100000)\\\\n\\\\ndef reproduce_reference_sample(data):\\\\n    from sklearn.model_selection import train_test_split\\\\n    rng = np.random.default_rng(2026)\\\\n    sample = pd.DataFrame()\\\\n    for chunk in read_chunks(data / \\\\\'train/train_source1.tsv\\\\\'):\\\\n        chunk = chunk[COLS].copy()\\\\n        chunk[\\\\\'_priority\\\\\'] = rng.random(len(chunk))\\\\n        sample = pd.concat([sample, chunk], ignore_index=True).nsmallest(4000, \\\\\'_priority\\\\\')\\\\n    sample = sample.drop(columns=\\\\\'_priority\\\\\').reset_index(drop=True)\\\\n    assert len(sample) == 4000 and sample.entity_id.is_unique\\\\n    ids = set(sample.entity_id)\\\\n    truth = {}\\\\n    for chunk in read_chunks(data / \\\\\'train/train_ground_truth.tsv\\\\\', [\\\\\'source1_entity_id\\\\\', \\\\\'matched_entity_ids\\\\\']):\\\\n        for row in chunk[chunk.source1_entity_id.isin(ids)].itertuples(index=False):\\\\n            truth[row.source1_entity_id] = set((x.strip() for x in row.matched_entity_ids.split(\\\\\',\\\\\') if x.strip()))\\\\n    assert set(truth) == ids\\\\n    strata = sample.country + \\\\\'|\\\\\' + sample.entity_id.map(lambda x: str(not truth[x]))\\\\n    train_idx, held = train_test_split(np.arange(4000), test_size=0.25, random_state=2026, stratify=strata)\\\\n    tune_idx, eval_idx = train_test_split(held, test_size=0.5, random_state=2027, stratify=strata.iloc[held])\\\\n    sample[\\\\\'split\\\\\'] = \\\\\'train\\\\\'\\\\n    sample.loc[tune_idx, \\\\\'split\\\\\'] = \\\\\'tune\\\\\'\\\\n    sample.loc[eval_idx, \\\\\'split\\\\\'] = \\\\\'evaluation\\\\\'\\\\n    return (sample, truth)\\\\n\\\\ndef clean_embedding_text(s):\\\\n    return \\\\\' \\\\\'.join(unicodedata.normalize(\\\\\'NFKC\\\\\', str(s)).split())\\\\n\\\\ndef text_view(frame, view):\\\\n    if view == \\\\\'name\\\\\':\\\\n        return [\\\\\'query: \\\\\' + clean_embedding_text(x) for x in frame.business_name]\\\\n    return [\\\\\'query: business name: \\\\\' + clean_embedding_text(n) + \\\\\'; address: \\\\\' + clean_embedding_text(a) for n, a in zip(frame.business_name, frame.business_address)]\\\\n\\\\nMODEL_ID=\\\\\'intfloat/multilingual-e5-small\\\\\'\\\\nREVISION=\\\\\'614241f622f53c4eeff9890bdc4f31cfecc418b3\\\\\'\\\\nDIM=384\\\\nVERSION=\\\\\'full-pool-e5-v1\\\\\'\\\\n\\\\ndef atomic_json(path, value):\\\\n    tmp=Path(str(path)+\\\\\'.tmp\\\\\')\\\\n    tmp.write_text(json.dumps(value,indent=2))\\\\n    os.replace(tmp,path)\\\\n\\\\ndef initialize(data,work):\\\\n    work=Path(work); data=Path(data); work.mkdir(parents=True,exist_ok=True)\\\\n    files=[data/f\\\\\'train/train_source{i}.tsv\\\\\' for i in (1,2,3)]+[data/\\\\\'train/train_ground_truth.tsv\\\\\']\\\\n    for p in files:\\\\n        assert p.exists(),f\\\\\'Missing dataset: {p}\\\\\'\\\\n    config={\\\\\'version\\\\\':VERSION,\\\\\'model\\\\\':MODEL_ID,\\\\\'revision\\\\\':REVISION,\\\\\'dim\\\\\':DIM,\\\\n            \\\\\'max_length\\\\\':256,\\\\\'text\\\\\':\\\\\'query: business name: NAME; address: ADDRESS\\\\\',\\\\n            \\\\\'files\\\\\':[(str(p),p.stat().st_size) for p in files],\\\\n            \\\\\'nlist_max\\\\\':2048,\\\\\'pq_m\\\\\':48,\\\\\'pq_bits\\\\\':8,\\\\\'nprobe\\\\\':64,\\\\\'overfetch\\\\\':200,\\\\n            \\\\\'candidate_k\\\\\':20,\\\\\'sample_seed\\\\\':2026,\\\\\'sample_size\\\\\':4000}\\\\n    cp=work/\\\\\'configuration.json\\\\\'\\\\n    if cp.exists():\\\\n        assert json.loads(cp.read_text())==json.loads(json.dumps(config)), \\\\\'Configuration changed. Use a new WORK folder.\\\\\'\\\\n    else: atomic_json(cp,config)\\\\n    log(f\\\\\'Free disk: {shutil.disk_usage(work).free/2**30:.1f} GiB\\\\\')\\\\n    return data,work\\\\n\\\\nclass Encoder:\\\\n    def __init__(self,batch_size=64):\\\\n        import torch\\\\n        from transformers import AutoTokenizer,AutoModel\\\\n        assert torch.cuda.is_available(),\\\\\'Enable a GPU in Kaggle Settings, then restart/run all.\\\\\'\\\\n        self.torch=torch; self.batch_size=batch_size\\\\n        self.tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False)\\\\n        self.model=AutoModel.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False,\\\\n                                           use_safetensors=True).cuda().half().eval()\\\\n        log(\\\\\'GPU: \\\\\'+torch.cuda.get_device_name(0))\\\\n    def encode(self,frame):\\\\n        texts=text_view(frame,\\\\\'combined\\\\\'); parts=[]; torch=self.torch\\\\n        with torch.inference_mode():\\\\n            for start in range(0,len(texts),self.batch_size):\\\\n                inp=self.tokenizer(texts[start:start+self.batch_size],max_length=256,\\\\n                                   padding=True,truncation=True,return_tensors=\\\\\'pt\\\\\').to(\\\\\'cuda\\\\\')\\\\n                hidden=self.model(**inp).last_hidden_state.float()\\\\n                mask=inp[\\\\\'attention_mask\\\\\'].unsqueeze(-1)\\\\n                vec=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)\\\\n                vec=torch.nn.functional.normalize(vec,p=2,dim=1)\\\\n                parts.append(vec.cpu().numpy())\\\\n        return np.concatenate(parts).astype(\\\\\'float32\\\\\') if parts else np.empty((0,DIM),np.float32)\\\\n\\\\ndef open_records(work):\\\\n    db=sqlite3.connect(work/\\\\\'records.sqlite\\\\\')\\\\n    db.execute(\\\\\'PRAGMA cache_size=-65536\\\\\')\\\\n    db.execute(\\\\\'CREATE TABLE IF NOT EXISTS records (rid INTEGER PRIMARY KEY, entity_id TEXT, business_name TEXT, business_address TEXT, country TEXT)\\\\\')\\\\n    db.execute(\\\\\'CREATE TABLE IF NOT EXISTS progress (source INTEGER PRIMARY KEY, done INTEGER, complete INTEGER)\\\\\')\\\\n    db.commit()\\\\n    return db\\\\n\\\\ndef encode_pool(data,work,encoder,split=\\"train\\"):\\\\n    \\"\\"\\"Vectors flushed before SQLite transaction commits; uncommitted rows overwritten on restart.\\"\\"\\"\\\\n    manifest_path=work/\\\\\'pool_manifest.json\\\\\'\\\\n    if not manifest_path.exists():\\\\n        counts={}\\\\n        for source in (2,3):\\\\n            counts[str(source)]=sum(len(c) for c in read_chunks(data/f\\\\\'{split}/{split}_source{source}.tsv\\\\\',[\\\\\'entity_id\\\\\']))\\\\n        atomic_json(manifest_path,{\\\\\'counts\\\\\':counts,\\\\\'total\\\\\':sum(counts.values())})\\\\n    manifest=json.loads(manifest_path.read_text()); total=manifest[\\\\\'total\\\\\']\\\\n    vp=work/\\\\\'vectors.f16\\\\\'; needed=total*DIM*2\\\\n    db=open_records(work)\\\\n    exists=vp.exists()\\\\n    if not exists:\\\\n        assert db.execute(\\\\\'SELECT COUNT(*) FROM records\\\\\').fetchone()[0]==0,\\\\\'Vectors missing but database has progress. Restore both or use new WORK.\\\\\'\\\\n        # Includes space for metadata DB, compressed indexes, model cache and reserve.\\\\n        assert shutil.disk_usage(work).free>needed+5*2**30, \\\\\'Need about 13 GiB free before full-pool encoding. Use a fresh notebook.\\\\\'\\\\n    else: assert vp.stat().st_size==needed,\\\\\'Incomplete vector allocation. Use new WORK directory.\\\\\'\\\\n    vectors=np.memmap(vp,mode=\\\\\'r+\\\\\' if exists else \\\\\'w+\\\\\',dtype=np.float16,shape=(total,DIM))\\\\n    base=0; started=time.time(); encoded=0\\\\n    for source in (2,3):\\\\n        saved=db.execute(\\\\\'SELECT done,complete FROM progress WHERE source=?\\\\\',(source,)).fetchone()\\\\n        done=saved[0] if saved else 0\\\\n        if saved and saved[1]:\\\\n            log(f\\\\\'Source {source}: already encoded {done:,} rows\\\\\'); base+=manifest[\\\\\'counts\\\\\'][str(source)]; continue\\\\n        log(f\\\\\'Source {source}: resume at {done:,}; reading input from beginning to that position\\\\\')\\\\n        seen=0\\\\n        chunks=pd.read_csv(data/f\\\\\'{split}/{split}_source{source}.tsv\\\\\',sep=\\\\\'\\\\\\\\t\\\\\',dtype=str,\\\\n                           keep_default_na=False,usecols=COLS,chunksize=10000)\\\\n        for frame in chunks:\\\\n            end=seen+len(frame)\\\\n            if end<=done: seen=end; continue\\\\n            frame=frame.iloc[max(0,done-seen):][COLS]\\\\n            first=max(done,seen); batch=encoder.encode(frame)\\\\n            assert batch.shape==(len(frame),DIM) and np.isfinite(batch).all()\\\\n            vectors[base+first:base+end]=batch.astype(np.float16); vectors.flush()\\\\n            rows=[(base+first+j,str(r.entity_id),str(r.business_name),str(r.business_address),norm(r.country))\\\\n                  for j,r in enumerate(frame.itertuples(index=False))]\\\\n            with db:\\\\n                db.executemany(\\\\\'INSERT INTO records VALUES (?,?,?,?,?)\\\\\',rows)\\\\n                db.execute(\\\\\'INSERT OR REPLACE INTO progress VALUES (?,?,0)\\\\\',(source,end))\\\\n            encoded+=len(frame); seen=end; done=end\\\\n            if end%100000==0:\\\\n                rate=encoded/max(time.time()-started,1)\\\\n                remaining=total-(base+end)\\\\n                log(f\\\\\'Source {source}: {end:,}; {rate:.0f} rows/sec; remaining encoding ~{remaining/rate/60:.1f} min\\\\\')\\\\n                assert shutil.disk_usage(work).free>1.5*2**30,\\\\\'Low disk. Progress saved; free space before resuming.\\\\\'\\\\n        assert done==manifest[\\\\\'counts\\\\\'][str(source)]\\\\n        with db: db.execute(\\\\\'INSERT OR REPLACE INTO progress VALUES (?,?,1)\\\\\',(source,done))\\\\n        base+=done\\\\n    assert db.execute(\\\\\'SELECT COUNT(*) FROM records\\\\\').fetchone()[0]==total\\\\n    atomic_json(work/\\\\\'encoding_complete.json\\\\\',{\\\\\'total\\\\\':total})\\\\n    db.close(); del vectors\\\\n    log(f\\\\\'Encoding complete: {total:,} target records.\\\\\')\\\\n\\\\ndef build_indexes(work):\\\\n    \\"\\"\\"IVF-PQ index per country. Completed countries and trained quantizers are reusable.\\"\\"\\"\\\\n    import faiss\\\\n    faiss.omp_set_num_threads(min(4,os.cpu_count() or 2))\\\\n    total=json.loads((work/\\\\\'encoding_complete.json\\\\\').read_text())[\\\\\'total\\\\\']\\\\n    vectors=np.memmap(work/\\\\\'vectors.f16\\\\\',mode=\\\\\'r\\\\\',dtype=np.float16,shape=(total,DIM))\\\\n    db=open_records(work)\\\\n    countries=[r[0] for r in db.execute(\\\\\'SELECT DISTINCT country FROM records\\\\\')]\\\\n    catalog=[]\\\\n    for country in sorted(countries):\\\\n        token=hashlib.sha256(country.encode()).hexdigest()[:12]\\\\n        path=work/f\\\\\'index_{token}.faiss\\\\\'; trained=work/f\\\\\'trained_{token}.faiss\\\\\'\\\\n        count=db.execute(\\\\\'SELECT COUNT(*) FROM records WHERE country=?\\\\\',(country,)).fetchone()[0]\\\\n        catalog.append({\\\\\'country\\\\\':country,\\\\\'file\\\\\':path.name,\\\\\'count\\\\\':count})\\\\n        if path.exists():\\\\n            check=faiss.read_index(str(path)); assert check.ntotal==count\\\\n            del check; log(f\\\\\'{country}: completed index reused\\\\\'); continue\\\\n        ids=np.fromiter((r[0] for r in db.execute(\\\\\'SELECT rid FROM records WHERE country=? ORDER BY rid\\\\\',(country,))),dtype=np.int64,count=count)\\\\n        # At least 50 training vectors per IVF centroid, up to 100k training vectors.\\\\n        nlist=min(2048,max(1,count//50))\\\\n        if trained.exists(): index=faiss.read_index(str(trained))\\\\n        elif count<10000:\\\\n            index=faiss.IndexIDMap2(faiss.IndexFlatIP(DIM))\\\\n        else:\\\\n            index=faiss.IndexIVFPQ(faiss.IndexFlatIP(DIM),DIM,nlist,48,8,faiss.METRIC_INNER_PRODUCT)\\\\n            index.cp.seed=2026; index.pq.cp.seed=2026\\\\n            train_ids=np.random.default_rng(2026).choice(ids,size=min(count,110000),replace=False)\\\\n            train_vectors=np.asarray(vectors[np.sort(train_ids)],dtype=np.float32)\\\\n            faiss.normalize_L2(train_vectors)\\\\n            log(f\\\\\'{country}: training IVF-PQ on {len(train_vectors):,} vectors; {nlist} cells\\\\\')\\\\n            index.train(train_vectors); del train_vectors\\\\n            faiss.write_index(index,str(trained)+\\\\\'.tmp\\\\\'); os.replace(str(trained)+\\\\\'.tmp\\\\\',trained)\\\\n        started=time.time()\\\\n        # An interrupted country is re-added from its saved trained quantizer.\\\\n        for start in range(0,count,25000):\\\\n            batch_ids=ids[start:start+25000]\\\\n            x=np.asarray(vectors[batch_ids],dtype=np.float32); faiss.normalize_L2(x)\\\\n            index.add_with_ids(x,batch_ids)\\\\n            if start%250000==0: log(f\\\\\'{country}: indexed {min(start+25000,count):,}/{count:,}; {(time.time()-started)/60:.1f} min\\\\\')\\\\n        assert index.ntotal==count\\\\n        faiss.write_index(index,str(path)+\\\\\'.tmp\\\\\'); os.replace(str(path)+\\\\\'.tmp\\\\\',path)\\\\n        del index,ids; gc.collect()\\\\n    atomic_json(work/\\\\\'index_catalog.json\\\\\',catalog)\\\\n    db.close(); del vectors\\\\n    log(\\\\\'All country indexes ready.\\\\\')\\\\n\\\\ndef prepare_queries(data,work,encoder):\\\\n    sample_path=work/\\\\\'sample.pkl\\\\\'\\\\n    if sample_path.exists():\\\\n        with sample_path.open(\\\\\'rb\\\\\') as f: sample,truth=pickle.load(f)\\\\n    else:\\\\n        sample,truth=reproduce_reference_sample(data)\\\\n        with Path(str(sample_path)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f: pickle.dump((sample,truth),f)\\\\n        os.replace(str(sample_path)+\\\\\'.tmp\\\\\',sample_path)\\\\n        sample.to_csv(work/\\\\\'sample.tsv\\\\\',sep=\\\\\'\\\\\\\\t\\\\\',index=False)\\\\n    path=work/\\\\\'query_vectors.npy\\\\\'\\\\n    if path.exists(): q=np.load(path)\\\\n    else:\\\\n        q=encoder.encode(sample)\\\\n        with Path(str(path)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f: np.save(f,q)\\\\n        os.replace(str(path)+\\\\\'.tmp\\\\\',path)\\\\n    assert q.shape==(len(sample),DIM)\\\\n    log(f\\\\\'Sample splits: {sample.split.value_counts().to_dict()}\\\\\')\\\\n    return sample,truth,q\\\\n\\\\ndef retrieve(work,sample,q,nprobe=64,overfetch=200,save_k=50):\\\\n    import faiss\\\\n    query_key=hashlib.sha256((\\\\\'|\\\\\'.join(sample.entity_id)).encode()).hexdigest()[:12]\\\\n    cache=work/f\\\\\'retrieval_{query_key}_p{nprobe}_r{overfetch}_k{save_k}.npz\\\\\'\\\\n    if cache.exists():\\\\n        saved=np.load(cache); return saved[\\\\\'ids\\\\\'],saved[\\\\\'scores\\\\\']\\\\n    total=json.loads((work/\\\\\'encoding_complete.json\\\\\').read_text())[\\\\\'total\\\\\']\\\\n    vectors=np.memmap(work/\\\\\'vectors.f16\\\\\',mode=\\\\\'r\\\\\',dtype=np.float16,shape=(total,DIM))\\\\n    ids=np.full((len(q),save_k),-1,np.int64); scores=np.full((len(q),save_k),-1,np.float32)\\\\n    countries=sample.country.map(norm).to_numpy()\\\\n    for item in json.loads((work/\\\\\'index_catalog.json\\\\\').read_text()):\\\\n        qi=np.flatnonzero(countries==item[\\\\\'country\\\\\'])\\\\n        if not len(qi): continue\\\\n        index=faiss.read_index(str(work/item[\\\\\'file\\\\\']))\\\\n        if hasattr(index,\\\\\'nprobe\\\\\'): index.nprobe=min(nprobe,index.nlist)\\\\n        log(f\\"Searching {len(qi)} businesses in {item[\\\\\'country\\\\\']}: {index.ntotal:,} records\\")\\\\n        for start in range(0,len(qi),64):\\\\n            batch=qi[start:start+64]\\\\n            _,approx=index.search(np.ascontiguousarray(q[batch],dtype=np.float32),overfetch)\\\\n            for i,neighbors in zip(batch,approx):\\\\n                neighbors=np.unique(neighbors[neighbors>=0])\\\\n                if not len(neighbors): continue\\\\n                x=np.asarray(vectors[neighbors],dtype=np.float32)\\\\n                x/=np.maximum(np.linalg.norm(x,axis=1,keepdims=True),1e-12)\\\\n                exact=x@q[i]\\\\n                order=np.lexsort((neighbors,-exact))[:save_k]\\\\n                chosen=neighbors[order]; k=len(chosen)\\\\n                ids[i,:k]=chosen; scores[i,:k]=exact[order]\\\\n        del index; gc.collect()\\\\n    with Path(str(cache)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f: np.savez_compressed(f,ids=ids,scores=scores)\\\\n    os.replace(str(cache)+\\\\\'.tmp\\\\\',cache)\\\\n    return ids,scores\\\\n\\\\ndef fetch_records(work,ids):\\\\n    db=open_records(work); rows={}\\\\n    valid=np.unique(ids[ids>=0])\\\\n    for start in range(0,len(valid),800):\\\\n        batch=[int(x) for x in valid[start:start+800]]\\\\n        for r in db.execute(\\\\\'SELECT * FROM records WHERE rid IN (\\\\\'+\\\\\',\\\\\'.join(\\\\\'?\\\\\'*len(batch))+\\\\\')\\\\\',batch):\\\\n            rows[r[0]]=tuple(r[1:])\\\\n    db.close()\\\\n    assert len(rows)==len(valid)\\\\n    return rows\\\\n\\\\ndef retrieval_report(sample,truth,ids,records):\\\\n    out=[]\\\\n    for split in (\\\\\'train\\\\\',\\\\\'tune\\\\\',\\\\\'evaluation\\\\\'):\\\\n        indices=np.flatnonzero(sample.split.to_numpy()==split)\\\\n        if not len(indices): continue\\\\n        for k in (10,20,50):\\\\n            found=total=0; sizes=[]\\\\n            for i in indices:\\\\n                gt=truth[sample.iloc[i].entity_id]\\\\n                pred={records[int(r)][0] for r in ids[i,:k] if r>=0}\\\\n                found+=len(gt&pred); total+=len(gt); sizes.append(len(pred))\\\\n            out.append(dict(split=split,k=k,found=found,true_links=total,\\\\n                            recall=found/total if total else 1,mean_candidates=float(np.mean(sizes)),\\\\n                            p95_candidates=float(np.percentile(sizes,95)),max_candidates=max(sizes)))\\\\n    return pd.DataFrame(out)\\\\n\\\\nMATCH_FEATURES=FEATURES+[\\\\\'embedding_cosine\\\\\']\\\\ndef make_pairs(sample,truth,ids,scores,records,k=20):\\\\n    x=[]; y=[]; owners=[]; candidate_ids=[]\\\\n    for i,left in enumerate(sample[COLS].itertuples(index=False,name=None)):\\\\n        for rank,rid in enumerate(ids[i,:k]):\\\\n            if rid<0: continue\\\\n            right=records[int(rid)]\\\\n            x.append(features(left,right)+[float(scores[i,rank])])\\\\n            y.append(int(right[0] in truth[left[0]])); owners.append(i); candidate_ids.append(right[0])\\\\n    return np.asarray(x,np.float32),np.asarray(y,np.uint8),np.asarray(owners,np.int32),candidate_ids\\\\n\\\\ndef metrics(sample,truth,owners,candidate_ids,prob,threshold,split):\\\\n    selected=np.flatnonzero(sample.split.to_numpy()==split)\\\\n    pred={int(i):set() for i in selected}\\\\n    for i,c,p in zip(owners,candidate_ids,prob):\\\\n        if int(i) in pred and p>=threshold: pred[int(i)].add(c)\\\\n    scores=[]; tp=fp=fn=singletons=singleton_correct=0\\\\n    for i in selected:\\\\n        gt=truth[sample.iloc[i].entity_id]; pr=pred[int(i)]\\\\n        a=len(gt&pr); b=len(pr-gt); c=len(gt-pr)\\\\n        tp+=a; fp+=b; fn+=c\\\\n        if not gt:\\\\n            singletons+=1; singleton_correct+=int(not pr); scores.append(float(not pr))\\\\n        else: scores.append(1.25*a/(1.25*a+b+.25*c))\\\\n    return dict(macro_F05=float(np.mean(scores)),correct_accepted=tp,incorrect_accepted=fp,\\\\n                missed_true_links=fn,singletons_correct=singleton_correct,singletons=singletons)\\\\n\\\\ndef fit_and_evaluate(work,sample,truth,ids,scores,records):\\\\n    from importlib.metadata import version\\\\n    x,y,owners,candidate_ids=make_pairs(sample,truth,ids,scores,records,k=20)\\\\n    splits=sample.split.to_numpy()[owners]; train=splits==\\\\\'train\\\\\'\\\\n    assert len(np.unique(y[train]))==2,\\\\\'Need positive and negative training pairs.\\\\\'\\\\n    log(f\\\\\'Training new matcher: {train.sum():,} pairs, {y[train].sum():,} true matches\\\\\')\\\\n    model=HistGradientBoostingClassifier(learning_rate=.08,max_iter=250,max_leaf_nodes=31,\\\\n              l2_regularization=1.,early_stopping=False,random_state=2026)\\\\n    with threadpool_limits(limits=4):\\\\n        model.fit(x[train],y[train]); prob=model.predict_proba(x)[:,1]\\\\n    # Only tuning businesses choose the threshold. Evaluation is read once afterwards.\\\\n    tuning=[]\\\\n    for t in np.linspace(.05,.95,91):\\\\n        tuning.append({\\\\\'threshold\\\\\':float(t),**metrics(sample,truth,owners,candidate_ids,prob,t,\\\\\'tune\\\\\')})\\\\n    best=max(tuning,key=lambda d:(d[\\\\\'macro_F05\\\\\'],d[\\\\\'threshold\\\\\']))\\\\n    cutoff=best[\\\\\'threshold\\\\\']\\\\n    evaluation=metrics(sample,truth,owners,candidate_ids,prob,cutoff,\\\\\'evaluation\\\\\')\\\\n    selected=np.flatnonzero(sample.split.to_numpy()==\\\\\'evaluation\\\\\')\\\\n    search_missed=0\\\\n    for i in selected:\\\\n        retrieved={records[int(r)][0] for r in ids[i,:20] if r>=0}\\\\n        search_missed+=len(truth[sample.iloc[i].entity_id]-retrieved)\\\\n    evaluation[\\\\\'missed_by_search\\\\\']=search_missed\\\\n    evaluation[\\\\\'retrieved_but_rejected\\\\\']=evaluation[\\\\\'missed_true_links\\\\\']-search_missed\\\\n    result={\\\\\'version\\\\\':VERSION,\\\\\'model_id\\\\\':MODEL_ID,\\\\\'revision\\\\\':REVISION,\\\\\'candidate_k\\\\\':20,\\\\n            \\\\\'ann_nprobe\\\\\':64,\\\\\'ann_overfetch\\\\\':200,\\\\\'threshold\\\\\':cutoff,\\\\\'tuning\\\\\':best,\\\\n            \\\\\'evaluation\\\\\':evaluation,\\\\\'features\\\\\':MATCH_FEATURES,\\\\n            \\\\\'scope\\\\\':\\\\\'Full training S2/S3; 3000 S1 train, 500 tune, 500 evaluation; no test predictions.\\\\\',\\\\n            \\\\\'versions\\\\\':{p:version(p) for p in [\\\\\'numpy\\\\\',\\\\\'pandas\\\\\',\\\\\'scikit-learn\\\\\',\\\\\'faiss-cpu\\\\\',\\\\\'torch\\\\\',\\\\\'transformers\\\\\']}}\\\\n    atomic_json(work/\\\\\'experiment_results.json\\\\\',result)\\\\n    pd.DataFrame(tuning).to_csv(work/\\\\\'threshold_tuning.csv\\\\\',index=False)\\\\n    with (work/\\\\\'matcher.pkl\\\\\').open(\\\\\'wb\\\\\') as f: pickle.dump({\\\\\'model\\\\\':model,\\\\\'threshold\\\\\':cutoff,\\\\\'features\\\\\':MATCH_FEATURES},f)\\\\n    # Save diagnostics, clearly labeled TRAINING evaluation, not a portal submission.\\\\n    diag=pd.DataFrame({\\\\\'source1_id\\\\\':sample.entity_id.to_numpy()[owners],\\\\\'candidate_id\\\\\':candidate_ids,\\\\n                       \\\\\'split\\\\\':splits,\\\\\'label\\\\\':y,\\\\\'score\\\\\':prob})\\\\n    diag[diag.split!=\\\\\'train\\\\\'].to_csv(work/\\\\\'heldout_pair_scores.tsv\\\\\',sep=\\\\\'\\\\\\\\t\\\\\',index=False)\\\\n    log(json.dumps(result,indent=2))\\\\n    return result\\\\n\', \'submit_pipeline.py\': \'\\"\\"\\"Run the evaluated E5 + matcher pipeline on the complete challenge test set.\\\\n\\\\nNo fitting or threshold selection occurs here. Candidate exports contain exactly\\\\nall final candidate pairs passed to the saved matcher.\\\\n\\"\\"\\"\\\\nimport argparse, csv, gc, hashlib, json, os, pickle, shutil, subprocess, sys, time, zipfile\\\\nfrom pathlib import Path\\\\nimport numpy as np\\\\nimport pandas as pd\\\\nfrom threadpoolctl import threadpool_limits\\\\nimport embedding_core as core\\\\n\\\\nVERSION=\\\\\'embedding-deadline-v2\\\\\'\\\\nPREDICT_CHUNK=2000\\\\n\\\\ndef sha256(path):\\\\n    h=hashlib.sha256()\\\\n    with Path(path).open(\\\\\'rb\\\\\') as f:\\\\n        for block in iter(lambda:f.read(4*1024*1024),b\\\\\'\\\\\'):h.update(block)\\\\n    return h.hexdigest()\\\\n\\\\ndef locate_model(explicit=None):\\\\n    if explicit:\\\\n        candidates=[Path(explicit)]\\\\n    else:\\\\n        candidates=[]\\\\n        local=Path(\\\\\'/kaggle/working/full_pool_embedding_v1\\\\\')\\\\n        if (local/\\\\\'matcher.pkl\\\\\').exists():candidates.append(local)\\\\n        for path in Path(\\\\\'/kaggle/input\\\\\').rglob(\\\\\'experiment_results.json\\\\\'):\\\\n            parent=path.parent\\\\n            if (parent/\\\\\'matcher.pkl\\\\\').exists():candidates.append(parent)\\\\n    valid=[]\\\\n    for path in sorted(set(candidates)):\\\\n        result=path/\\\\\'experiment_results.json\\\\\'\\\\n        if result.exists() and (path/\\\\\'matcher.pkl\\\\\').exists():\\\\n            try:meta=json.loads(result.read_text())\\\\n            except (ValueError,OSError):continue\\\\n            if meta.get(\\\\\'version\\\\\')==\\\\\'full-pool-e5-v1\\\\\' and meta.get(\\\\\'model_id\\\\\')==core.MODEL_ID:valid.append(path)\\\\n    if len(valid)!=1:\\\\n        raise RuntimeError(\\\\\'Attach the completed full-pool experiment notebook OUTPUT as an input, \\\\\'\\\\n          \\\\\'then set MODEL_DIR to the folder containing matcher.pkl and experiment_results.json. \\\\\'\\\\n          f\\\\\'Found {len(valid)} matching folders: {[str(p) for p in valid]}\\\\\')\\\\n    return valid[0]\\\\n\\\\ndef load_model(model_dir):\\\\n    import sklearn\\\\n    meta=json.loads((model_dir/\\\\\'experiment_results.json\\\\\').read_text())\\\\n    assert meta[\\\\\'revision\\\\\']==core.REVISION and meta[\\\\\'features\\\\\']==core.MATCH_FEATURES\\\\n    assert meta[\\\\\'candidate_k\\\\\']==20 and (meta[\\\\\'ann_nprobe\\\\\'],meta[\\\\\'ann_overfetch\\\\\']) in [(64,200),(128,400)]\\\\n    assert sklearn.__version__==meta[\\\\\'versions\\\\\'][\\\\\'scikit-learn\\\\\'], (\\\\n        \\\\\'Install scikit-learn==\\\\\'+meta[\\\\\'versions\\\\\'][\\\\\'scikit-learn\\\\\']+\\\\\' before loading this model.\\\\\')\\\\n    # This must be YOUR trusted matcher produced by the previous notebook.\\\\n    with (model_dir/\\\\\'matcher.pkl\\\\\').open(\\\\\'rb\\\\\') as f:bundle=pickle.load(f)\\\\n    assert bundle[\\\\\'features\\\\\']==core.MATCH_FEATURES\\\\n    assert abs(float(bundle[\\\\\'threshold\\\\\'])-float(meta[\\\\\'threshold\\\\\']))<1e-10\\\\n    assert bundle[\\\\\'model\\\\\'].n_features_in_==len(core.MATCH_FEATURES)\\\\n    assert list(bundle[\\\\\'model\\\\\'].classes_)==[0,1]\\\\n    bundle[\\\\\'ann_nprobe\\\\\']=meta[\\\\\'ann_nprobe\\\\\'];bundle[\\\\\'ann_overfetch\\\\\']=meta[\\\\\'ann_overfetch\\\\\']\\\\n    return bundle,meta\\\\n\\\\ndef initialize(data,work,model_dir):\\\\n    data=Path(data);work=Path(work);model_dir=Path(model_dir)\\\\n    work.mkdir(parents=True,exist_ok=True)\\\\n    files=[data/f\\\\\'test/test_source{i}.tsv\\\\\' for i in (1,2,3)]\\\\n    validator=data.parent/\\\\\'utils/validate_submission.py\\\\\'\\\\n    for path in files+[validator]:\\\\n        if not path.exists():raise FileNotFoundError(path)\\\\n    bundle,meta=load_model(model_dir)\\\\n    core.log(\\\\\'Checking input fingerprints and saved model...\\\\\')\\\\n    config={\\\\\'version\\\\\':VERSION,\\\\\'inputs\\\\\':{p.name:sha256(p) for p in files},\\\\n      \\\\\'matcher_sha256\\\\\':sha256(model_dir/\\\\\'matcher.pkl\\\\\'),\\\\\'model_id\\\\\':core.MODEL_ID,\\\\n      \\\\\'revision\\\\\':core.REVISION,\\\\\'max_length\\\\\':256,\\\\\'dim\\\\\':core.DIM,\\\\n      \\\\\'candidate_k\\\\\':20,\\\\\'nprobe\\\\\':meta[\\\\\'ann_nprobe\\\\\'],\\\\\'overfetch\\\\\':meta[\\\\\'ann_overfetch\\\\\'],\\\\\'threshold\\\\\':float(bundle[\\\\\'threshold\\\\\']),\\\\n      \\\\\'prediction_chunk\\\\\':PREDICT_CHUNK,\\\\n      \\\\\'core_sha256\\\\\':sha256(Path(core.__file__)),\\\\\'runner_sha256\\\\\':sha256(Path(__file__))}\\\\n    cp=work/\\\\\'submission_config.json\\\\\'\\\\n    if cp.exists():assert json.loads(cp.read_text())==config,\\\\\'Inputs, model or code changed. Use a NEW WORK directory.\\\\\'\\\\n    else:core.atomic_json(cp,config)\\\\n    artifact=work/\\\\\'model\\\\\';artifact.mkdir(exist_ok=True)\\\\n    for name in [\\\\\'matcher.pkl\\\\\',\\\\\'experiment_results.json\\\\\']:\\\\n        dest=artifact/name\\\\n        if dest.resolve()!=(model_dir/name).resolve():shutil.copy2(model_dir/name,dest)\\\\n    core.log(f\\"Saved matcher cutoff: {bundle[\\\\\'threshold\\\\\']:.3f}; 20 candidates; {meta[\\\\\'ann_nprobe\\\\\']} cells / {meta[\\\\\'ann_overfetch\\\\\']} intermediate neighbors\\")\\\\n    return bundle,meta,config\\\\n\\\\nclass TestSearcher:\\\\n    def __init__(self,work,nprobe=64,overfetch=200):\\\\n        import faiss\\\\n        self.work=work;self.faiss=faiss;self.overfetch=overfetch\\\\n        self.total=json.loads((work/\\\\\'encoding_complete.json\\\\\').read_text())[\\\\\'total\\\\\']\\\\n        self.vectors=np.memmap(work/\\\\\'vectors.f16\\\\\',mode=\\\\\'r\\\\\',dtype=np.float16,shape=(self.total,core.DIM))\\\\n        self.db=sqlite3.connect(f\\\\\'file:{(work/\\"records.sqlite\\").resolve()}?mode=ro\\\\\',uri=True)\\\\n        self.db.execute(\\\\\'PRAGMA cache_size=-65536\\\\\')\\\\n        self.indexes={}\\\\n        for item in json.loads((work/\\\\\'index_catalog.json\\\\\').read_text()):\\\\n            index=faiss.read_index(str(work/item[\\\\\'file\\\\\']))\\\\n            assert index.ntotal==item[\\\\\'count\\\\\']\\\\n            if hasattr(index,\\\\\'nprobe\\\\\'):index.nprobe=min(nprobe,index.nlist)\\\\n            self.indexes[item[\\\\\'country\\\\\']]=index\\\\n    def search(self,frame,q):\\\\n        \\"\\"\\"Same approximate search and float16-vector cosine reranking as evaluation.\\"\\"\\"\\\\n        ids=np.full((len(frame),20),-1,np.int64)\\\\n        scores=np.full((len(frame),20),-1,np.float32)\\\\n        countries=frame.country.map(core.norm).to_numpy()\\\\n        for country in np.unique(countries):\\\\n            qi=np.flatnonzero(countries==country)\\\\n            index=self.indexes.get(country)\\\\n            if index is None:continue\\\\n            for start in range(0,len(qi),64):\\\\n                indices=qi[start:start+64]\\\\n                _,approx=index.search(np.ascontiguousarray(q[indices],dtype=np.float32),self.overfetch)\\\\n                # Batch gathers reduce Python/disk overhead; re-ranking remains cosine.\\\\n                valid=approx>=0\\\\n                safe=np.maximum(approx,0)\\\\n                x=np.asarray(self.vectors[safe],dtype=np.float32)\\\\n                x/=np.maximum(np.linalg.norm(x,axis=2,keepdims=True),1e-12)\\\\n                cos=np.einsum(\\\\\'bkd,bd->bk\\\\\',x,q[indices],optimize=False)\\\\n                cos[~valid]=-np.inf\\\\n                for local,i in enumerate(indices):\\\\n                    positions=np.flatnonzero(valid[local])\\\\n                    if not len(positions):continue\\\\n                    order=np.lexsort((approx[local,positions],-cos[local,positions]))\\\\n                    chosen=positions[order[:20]]\\\\n                    ids[i,:len(chosen)]=approx[local,chosen]\\\\n                    scores[i,:len(chosen)]=cos[local,chosen]\\\\n                del x\\\\n        return ids,scores\\\\n    def fetch(self,ids):\\\\n        valid=np.unique(ids[ids>=0]);rows={}\\\\n        for start in range(0,len(valid),800):\\\\n            batch=[int(x) for x in valid[start:start+800]]\\\\n            sql=\\\\\'SELECT * FROM records WHERE rid IN (\\\\\'+\\\\\',\\\\\'.join(\\\\\'?\\\\\'*len(batch))+\\\\\')\\\\\'\\\\n            for row in self.db.execute(sql,batch):rows[row[0]]=tuple(row[1:])\\\\n        assert len(rows)==len(valid),\\\\\'Candidate IDs missing from test record store.\\\\\'\\\\n        return rows\\\\n    def close(self):\\\\n        self.db.close();self.indexes.clear();del self.vectors\\\\n\\\\n# sqlite is separate from core to keep this module directly runnable.\\\\nimport sqlite3\\\\n\\\\ndef score_candidates(frame,ids,scores,records,bundle):\\\\n    candidates=[[] for _ in range(len(frame))];accepted=[[] for _ in range(len(frame))]\\\\n    features=[];owners=[];pair_ids=[]\\\\n    for i,left in enumerate(frame[core.COLS].itertuples(index=False,name=None)):\\\\n        seen=set()\\\\n        for rank,rid in enumerate(ids[i]):\\\\n            if rid<0:continue\\\\n            right=records[int(rid)]\\\\n            assert core.norm(left[3])==core.norm(right[3]),\\\\\'Country mismatch in retrieval.\\\\\'\\\\n            assert right[0].startswith((\\\\\'S2-\\\\\',\\\\\'S3-\\\\\'))\\\\n            assert right[0] not in seen,\\\\\'Duplicate candidate entity ID in test input/retrieval.\\\\\'\\\\n            seen.add(right[0]);candidates[i].append(right[0])\\\\n            features.append(core.features(left,right)+[float(scores[i,rank])])\\\\n            owners.append(i);pair_ids.append(right[0])\\\\n    if features:\\\\n        x=np.asarray(features,np.float32)\\\\n        assert np.isfinite(x).all()\\\\n        prob=bundle[\\\\\'model\\\\\'].predict_proba(x)[:,1]\\\\n        assert np.isfinite(prob).all()\\\\n        for owner,cid,score in zip(owners,pair_ids,prob):\\\\n            if score>=bundle[\\\\\'threshold\\\\\']:accepted[owner].append(cid)\\\\n    assert sum(map(len,candidates))==len(features)\\\\n    for cand,match in zip(candidates,accepted):\\\\n        assert len(cand)<=20 and set(match)<=set(cand)\\\\n    return candidates,accepted\\\\n\\\\ndef write_shard(path,frame,candidates,accepted,start):\\\\n    staging=path.with_name(path.name+\\\\\'.tmp\\\\\');staging.mkdir(parents=True,exist_ok=True)\\\\n    for name,values,col in [(\\\\\'candidate_pairs.tsv\\\\\',candidates,\\\\\'candidate_entity_ids\\\\\'),\\\\n                            (\\\\\'matching_results.tsv\\\\\',accepted,\\\\\'matched_entity_ids\\\\\')]:\\\\n        with (staging/name).open(\\\\\'w\\\\\',encoding=\\\\\'utf-8\\\\\',newline=\\\\\'\\\\\') as f:\\\\n            writer=csv.writer(f,delimiter=\\\\\'\\\\\\\\t\\\\\',lineterminator=\\\\\'\\\\\\\\n\\\\\')\\\\n            writer.writerow([\\\\\'source1_entity_id\\\\\',col])\\\\n            for sid,links in zip(frame.entity_id,values):writer.writerow([sid,\\\\\',\\\\\'.join(links)])\\\\n            f.flush();os.fsync(f.fileno())\\\\n    stats={\\\\\'start\\\\\':start,\\\\\'rows\\\\\':len(frame),\\\\\'first\\\\\':str(frame.entity_id.iloc[0]),\\\\\'last\\\\\':str(frame.entity_id.iloc[-1]),\\\\n       \\\\\'candidate_pairs\\\\\':sum(map(len,candidates)),\\\\\'predicted_links\\\\\':sum(map(len,accepted)),\\\\n       \\\\\'empty_candidates\\\\\':sum(not x for x in candidates),\\\\\'empty_matches\\\\\':sum(not x for x in accepted),\\\\n       \\\\\'candidate_histogram\\\\\':np.bincount([len(x) for x in candidates],minlength=21).tolist()}\\\\n    core.atomic_json(staging/\\\\\'stats.json\\\\\',stats)\\\\n    os.replace(staging,path)\\\\n    return stats\\\\n\\\\ndef predict(data,work,bundle,encoder):\\\\n    shards=work/\\\\\'shards\\\\\';shards.mkdir(exist_ok=True)\\\\n    searcher=TestSearcher(work,bundle.get(\\"ann_nprobe\\",64),bundle.get(\\"ann_overfetch\\",200))\\\\n    counts={};start=0;started=time.time();new=0\\\\n    total_s1=sum(len(c) for c in core.read_chunks(data/\\\\\'test/test_source1.tsv\\\\\',[\\\\\'entity_id\\\\\']))\\\\n    core.log(f\\\\\'Predicting all {total_s1:,} test Source 1 businesses; output checkpoint every {PREDICT_CHUNK:,}.\\\\\')\\\\n    try:\\\\n        with threadpool_limits(limits=4):\\\\n            searcher.faiss.omp_set_num_threads(min(4,os.cpu_count() or 2))\\\\n            for frame in pd.read_csv(data/\\\\\'test/test_source1.tsv\\\\\',sep=\\\\\'\\\\\\\\t\\\\\',dtype=str,\\\\n                                      keep_default_na=False,usecols=core.COLS,chunksize=PREDICT_CHUNK):\\\\n                frame=frame[core.COLS].reset_index(drop=True)\\\\n                path=shards/f\\\\\'{start:010d}\\\\\'\\\\n                if path.exists():\\\\n                    stats=json.loads((path/\\\\\'stats.json\\\\\').read_text())\\\\n                    assert stats[\\\\\'start\\\\\']==start and stats[\\\\\'rows\\\\\']==len(frame)\\\\n                    assert stats[\\\\\'first\\\\\']==frame.entity_id.iloc[0] and stats[\\\\\'last\\\\\']==frame.entity_id.iloc[-1]\\\\n                    assert all((path/n).exists() for n in [\\\\\'candidate_pairs.tsv\\\\\',\\\\\'matching_results.tsv\\\\\'])\\\\n                else:\\\\n                    if shutil.disk_usage(work).free<1.5*2**30:raise RuntimeError(\\\\\'Low disk. Completed shards saved; free space before rerunning.\\\\\')\\\\n                    q=encoder.encode(frame)\\\\n                    ids,scores=searcher.search(frame,q)\\\\n                    records=searcher.fetch(ids)\\\\n                    cand,accepted=score_candidates(frame,ids,scores,records,bundle)\\\\n                    stats=write_shard(path,frame,cand,accepted,start)\\\\n                    new+=len(frame)\\\\n                    del records,q,ids,scores,cand,accepted\\\\n                    if new==len(frame) or (start+len(frame))%10000==0:\\\\n                        rate=new/max(time.time()-started,1)\\\\n                        eta=(total_s1-start-len(frame))/max(rate,1)/60\\\\n                        core.log(f\\\\\'{start+len(frame):,}/{total_s1:,} businesses; {rate:.1f}/sec; remaining prediction ~{eta:.1f} min\\\\\')\\\\n                for k in [\\\\\'rows\\\\\',\\\\\'candidate_pairs\\\\\',\\\\\'predicted_links\\\\\',\\\\\'empty_candidates\\\\\',\\\\\'empty_matches\\\\\']:\\\\n                    counts[k]=counts.get(k,0)+stats[k]\\\\n                start+=len(frame)\\\\n    finally:searcher.close();gc.collect()\\\\n    assert start==total_s1\\\\n    core.atomic_json(work/\\\\\'prediction_complete.json\\\\\',counts)\\\\n    core.log(\\\\\'All test businesses predicted: \\\\\'+json.dumps(counts))\\\\n    return counts\\\\n\\\\ndef merge_outputs(data,work):\\\\n    counts=json.loads((work/\\\\\'prediction_complete.json\\\\\').read_text())\\\\n    out=work/\\\\\'output\\\\\';out.mkdir(exist_ok=True)\\\\n    paths=sorted(p for p in (work/\\\\\'shards\\\\\').iterdir() if p.is_dir() and p.name.isdigit())\\\\n    expected=0;hist=np.zeros(21,np.int64)\\\\n    for p in paths:\\\\n        s=json.loads((p/\\\\\'stats.json\\\\\').read_text())\\\\n        assert s[\\\\\'start\\\\\']==expected\\\\n        expected+=s[\\\\\'rows\\\\\'];hist+=s[\\\\\'candidate_histogram\\\\\']\\\\n    assert expected==counts[\\\\\'rows\\\\\']\\\\n    for filename,second in [(\\\\\'matching_results.tsv\\\\\',\\\\\'matched_entity_ids\\\\\'),(\\\\\'candidate_pairs.tsv\\\\\',\\\\\'candidate_entity_ids\\\\\')]:\\\\n        path=out/filename;tmp=Path(str(path)+\\\\\'.tmp\\\\\');rows=0\\\\n        with tmp.open(\\\\\'wb\\\\\') as f:\\\\n            header=f\\\\\'source1_entity_id\\\\\\\\t{second}\\\\\\\\n\\\\\'.encode()\\\\n            f.write(header)\\\\n            for p in paths:\\\\n                with (p/filename).open(\\\\\'rb\\\\\') as src:\\\\n                    assert src.readline()==header\\\\n                    for line in src:\\\\n                        f.write(line);rows+=1\\\\n        assert rows==expected\\\\n        os.replace(tmp,path)\\\\n    counts[\\\\\'mean_candidates\\\\\']=counts[\\\\\'candidate_pairs\\\\\']/counts[\\\\\'rows\\\\\']\\\\n    counts[\\\\\'candidate_histogram\\\\\']=hist.tolist()\\\\n    core.atomic_json(out/\\\\\'submission_summary.json\\\\\',counts)\\\\n    core.log(\\\\\'Merged submission files: \\\\\'+json.dumps(counts))\\\\n    return out\\\\n\\\\ndef validate(data,work):\\\\n    validator=data.parent/\\\\\'utils/validate_submission.py\\\\\'\\\\n    out=work/\\\\\'output\\\\\'\\\\n    command=[sys.executable,str(validator),\\\\\'--matching\\\\\',str(out/\\\\\'matching_results.tsv\\\\\'),\\\\n             \\\\\'--candidate\\\\\',str(out/\\\\\'candidate_pairs.tsv\\\\\'),\\\\\'--test-dir\\\\\',str(data/\\\\\'test\\\\\'),\\\\\'--check-ids\\\\\']\\\\n    core.log(\\\\\'Running official validator, including test ID existence checks...\\\\\')\\\\n    with (out/\\\\\'validation.log\\\\\').open(\\\\\'w\\\\\') as log:\\\\n        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)\\\\n        for line in proc.stdout:\\\\n            print(line,end=\\\\\'\\\\\',flush=True);log.write(line);log.flush()\\\\n        code=proc.wait()\\\\n    if code:raise RuntimeError(f\\\\\'Official validator failed (exit {code}). Do not submit. See validation.log.\\\\\')\\\\n    core.atomic_json(out/\\\\\'validated.json\\\\\',{\\\\\'matching_sha256\\\\\':sha256(out/\\\\\'matching_results.tsv\\\\\'),\\\\n               \\\\\'candidate_sha256\\\\\':sha256(out/\\\\\'candidate_pairs.tsv\\\\\'),\\\\\'check_ids\\\\\':True})\\\\n    core.log(\\\\\'VALIDATOR PASSED. Portal upload: \\\\\'+str(out/\\\\\'matching_results.tsv\\\\\'))\\\\n\\\\ndef package(work,meta):\\\\n    from importlib.metadata import version\\\\n    out=work/\\\\\'output\\\\\';assert (out/\\\\\'validated.json\\\\\').exists()\\\\n    validation=json.loads((out/\\\\\'validated.json\\\\\').read_text())\\\\n    assert validation[\\\\\'matching_sha256\\\\\']==sha256(out/\\\\\'matching_results.tsv\\\\\')\\\\n    assert validation[\\\\\'candidate_sha256\\\\\']==sha256(out/\\\\\'candidate_pairs.tsv\\\\\')\\\\n    (work/\\\\\'requirements.txt\\\\\').write_text(\\\\\'\\\\\\\\n\\\\\'.join(f\\\\\'{p}=={version(p)}\\\\\' for p in\\\\n             [\\\\\'torch\\\\\',\\\\\'transformers\\\\\',\\\\\'numpy\\\\\',\\\\\'pandas\\\\\',\\\\\'scikit-learn\\\\\',\\\\\'faiss-cpu\\\\\',\\\\\'safetensors\\\\\',\\\\\'sentencepiece\\\\\'])+\\\\\'\\\\\\\\n\\\\\')\\\\n    (work/\\\\\'README.md\\\\\').write_text(\\\\\'\\\\\'\\\\\'# Embedding submission v2\\\\nUpload `output/matching_results.tsv` to the leaderboard portal.\\\\nRetain `output/candidate_pairs.tsv`, the model, code, and methodology for final review.\\\\nThe candidate file contains exactly every final pair evaluated by the classifier.\\\\n\\\\n## Reproduce predictions\\\\nInstall requirements in a compatible GPU environment. Public E5 weights are pinned by revision.\\\\nRun: `python submit_pipeline.py --data /path/to/student_resource/dataset --model-dir ./model --work ./rerun --stage all`\\\\nThe input dataset and official validator must be provided separately. No business lookups occur.\\\\nAll test S1 entities receive one row, including empty predictions. Country matching is enforced.\\\\n\\\\n## Reproduce training\\\\nRun `training_experiment.ipynb` on the supplied training dataset to recreate the sampled\\\\ntraining experiment, matcher and cutoff. Its dataset and working paths are configurable.\\\\nThe trained matcher in `model/matcher.pkl` comes from that experiment, not test labels.\\\\nThe upgrade training notebook requires the v1 training caches as a read-only input.\\\\nOnly load your own trusted pickle models.\\\\n\\\\n## Checkpoints\\\\nThis archive excludes large vector and index caches. The complete saved Kaggle output\\\\ncontains those, plus completed prediction shards. Rerunning against an unchanged working\\\\nfolder resumes completed work. A discarded unsaved session cannot be resumed.\\\\n\\\\n## Final package\\\\nThis archive preserves submission evidence and runnable code. Before final challenge\\\\nclosure, transfer methodology into the organizer\\\\\'s Documentation_template.md and check\\\\nany organizer-specific archive naming or additional requirements.\\\\n\\\\\'\\\\\'\\\\\')\\\\n    (work/\\\\\'methodology.md\\\\\').write_text(f\\\\\'\\\\\'\\\\\'# Methodology \\u2014 embedding submission v2\\\\nNames and addresses are NFKC/whitespace normalized, preserving Unicode. E5 encodes\\\\n`query: business name: NAME; address: ADDRESS` with attention-masked mean pooling and\\\\nL2 normalization, maximum 256 tokens. Model: {core.MODEL_ID}; revision: {core.REVISION}.\\\\nPretrained weights are fixed, not fine-tuned. Business data comes only from the challenge.\\\\n\\\\nTest S2/S3 vectors form independent country-specific IVF-PQ indexes (up to 2048 cells,\\\\n48 subquantizers, 8 bits each). Quantizers use up to 110000 unlabeled target vectors\\\\nper country, seed 2026. Search probes {meta[\\\\\'ann_nprobe\\\\\']} cells, retrieves up to {meta[\\\\\'ann_overfetch\\\\\']} approximate candidates,\\\\nthen reranks by cosine against stored float16 vectors and keeps at most 20.\\\\nOnly these final 20 are passed to the matcher and written to candidate_pairs.tsv.\\\\nIntermediate retrieval counts and code are disclosed here for auditability.\\\\n\\\\nThe HistGradientBoostingClassifier uses the 18 lexical features defined in embedding_core.py\\\\nand embedding cosine similarity. It was trained on 30000 sampled S1 businesses (up to 600000 pairs),\\\\nwith 2000 tuning and 2000 fresh evaluation businesses. Threshold: {meta[\\\\\'threshold\\\\\']}.\\\\nTraining targets include the entire provided train S2/S3 pool. No truth matches are injected.\\\\nLocal evaluation macro F0.5: {meta[\\\\\'evaluation\\\\\'][\\\\\'macro_F05\\\\\']:.6f}; this is not a test leaderboard score.\\\\nTest predictions use the saved model and cutoff without retraining or threshold changes.\\\\n\\\\nSource 1 is processed in batches and files are assembled in input order. The official\\\\nvalidator runs with --check-ids. Empty match lists are retained. France is present in test\\\\nbut absent from labeled training; country-specific French accuracy has not been validated.\\\\nThe country index code derives countries dynamically, including France.\\\\n\\\\nSee output/submission_summary.json for actual candidate counts, and requirements.txt for versions.\\\\n\\\\\'\\\\\'\\\\\')\\\\n    # A small ZIP is easy to download even if the browser previews TSV files.\\\\n    with zipfile.ZipFile(out/\\\\\'matching_results_download.zip\\\\\',\\\\\'w\\\\\',zipfile.ZIP_DEFLATED) as z:\\\\n        z.write(out/\\\\\'matching_results.tsv\\\\\',\\\\\'matching_results.tsv\\\\\')\\\\n    archive=work.parent/\\\\\'embedding_submission_v2_backup.zip\\\\\'\\\\n    members=[\\\\\'README.md\\\\\',\\\\\'methodology.md\\\\\',\\\\\'requirements.txt\\\\\',\\\\\'submission_config.json\\\\\',\\\\n      \\\\\'embedding_core.py\\\\\',\\\\\'submit_pipeline.py\\\\\',\\\\\'training_experiment.ipynb\\\\\',\\\\n      \\\\\'model/matcher.pkl\\\\\',\\\\\'model/experiment_results.json\\\\\',\\\\\'output/matching_results.tsv\\\\\',\\\\n      \\\\\'output/candidate_pairs.tsv\\\\\',\\\\\'output/submission_summary.json\\\\\',\\\\\'output/validation.log\\\\\',\\\\\'output/validated.json\\\\\']\\\\n    for name in members:\\\\n        assert (work/name).exists(),f\\\\\'Missing backup member: {name}\\\\\'\\\\n    with zipfile.ZipFile(archive,\\\\\'w\\\\\',zipfile.ZIP_DEFLATED) as z:\\\\n        for name in members:z.write(work/name,name)\\\\n    core.log(\\\\\'Downloadable matches ZIP: \\\\\'+str(out/\\\\\'matching_results_download.zip\\\\\'))\\\\n    core.log(\\\\\'Retain backup: \\\\\'+str(archive))\\\\n    return archive\\\\n\\\\ndef main():\\\\n    parser=argparse.ArgumentParser()\\\\n    parser.add_argument(\\\\\'--data\\\\\',type=Path,required=True)\\\\n    parser.add_argument(\\\\\'--work\\\\\',type=Path,required=True)\\\\n    parser.add_argument(\\\\\'--model-dir\\\\\',type=Path,required=True)\\\\n    parser.add_argument(\\\\\'--stage\\\\\',choices=[\\\\\'all\\\\\',\\\\\'encode\\\\\',\\\\\'index\\\\\',\\\\\'predict\\\\\',\\\\\'finalize\\\\\'],default=\\\\\'all\\\\\')\\\\n    parser.add_argument(\\\\\'--batch-size\\\\\',type=int,default=64)\\\\n    args=parser.parse_args();data,work=args.data,args.work\\\\n    bundle,meta,config=initialize(data,work,args.model_dir)\\\\n    if args.stage in (\\\\\'all\\\\\',\\\\\'encode\\\\\'):\\\\n        encoder=core.Encoder(args.batch_size)\\\\n        core.encode_pool(data,work,encoder,split=\\\\\'test\\\\\')\\\\n        del encoder;gc.collect()\\\\n        import torch;torch.cuda.empty_cache()\\\\n    if args.stage in (\\\\\'all\\\\\',\\\\\'index\\\\\'):core.build_indexes(work)\\\\n    if args.stage in (\\\\\'all\\\\\',\\\\\'predict\\\\\'):\\\\n        encoder=core.Encoder(args.batch_size)\\\\n        predict(data,work,bundle,encoder)\\\\n        del encoder;gc.collect()\\\\n        import torch;torch.cuda.empty_cache()\\\\n    if args.stage in (\\\\\'all\\\\\',\\\\\'finalize\\\\\'):\\\\n        merge_outputs(data,work);validate(data,work);package(work,meta)\\\\n\\\\nif __name__==\\\\\'__main__\\\\\':main()\\\\n\', \'upgrade_train.py\': \'\\"\\"\\"Reuse read-only training embeddings; train 30k, tune 2k, evaluate 2k fresh S1 entities.\\"\\"\\"\\\\nimport os,sys,json,pickle,hashlib,time,gc,argparse\\\\nfrom pathlib import Path\\\\nfrom datetime import datetime,timezone\\\\nfrom zoneinfo import ZoneInfo\\\\nimport numpy as np,pandas as pd\\\\nfrom sklearn.model_selection import train_test_split\\\\nfrom sklearn.ensemble import HistGradientBoostingClassifier\\\\nfrom threadpoolctl import threadpool_limits\\\\nimport embedding_core as c\\\\nimport submit_pipeline as s\\\\n\\\\nSEED=20260927\\\\nTRAIN_N=30000\\\\nTUNE_N=2000\\\\nEVAL_N=2000\\\\nPROFILES={\\\\\'standard\\\\\':(64,200),\\\\\'strong\\\\\':(128,400)}\\\\n\\\\ndef resolve_cache(explicit=None):\\\\n    if explicit:candidates=[Path(explicit)]\\\\n    else:\\\\n        candidates=[x.parent for x in Path(\\\\\'/kaggle/input\\\\\').rglob(\\\\\'encoding_complete.json\\\\\')]\\\\n        local=Path(\\\\\'/kaggle/working/full_pool_embedding_v1\\\\\')\\\\n        if local.exists():candidates.append(local)\\\\n    valid=[]\\\\n    for p in sorted(set(candidates)):\\\\n        required=[\\\\\'vectors.f16\\\\\',\\\\\'records.sqlite\\\\\',\\\\\'index_catalog.json\\\\\',\\\\\'encoding_complete.json\\\\\',\\\\n                  \\\\\'sample.tsv\\\\\',\\\\\'matcher.pkl\\\\\',\\\\\'experiment_results.json\\\\\',\\\\\'configuration.json\\\\\']\\\\n        if not all((p/n).exists() for n in required):continue\\\\n        meta=json.loads((p/\\\\\'experiment_results.json\\\\\').read_text())\\\\n        if meta.get(\\\\\'version\\\\\')!=\\\\\'full-pool-e5-v1\\\\\':continue\\\\n        assert meta[\\\\\'model_id\\\\\']==c.MODEL_ID and meta[\\\\\'revision\\\\\']==c.REVISION\\\\n        count=json.loads((p/\\\\\'encoding_complete.json\\\\\').read_text())[\\\\\'total\\\\\']\\\\n        assert (p/\\\\\'vectors.f16\\\\\').stat().st_size==count*c.DIM*2\\\\n        catalog=json.loads((p/\\\\\'index_catalog.json\\\\\').read_text())\\\\n        assert sum(x[\\\\\'count\\\\\'] for x in catalog)==count\\\\n        assert all((p/x[\\\\\'file\\\\\']).exists() for x in catalog)\\\\n        valid.append(p)\\\\n    if len(valid)!=1:\\\\n        raise RuntimeError(\\\\\'Attach the FULL saved v1 notebook output as input, not only the small results ZIP. \\\\\'\\\\n          \\\\\'Set CACHE_DIR to its full_pool_embedding_v1 folder if more than one is found. Found: \\\\\'+str(valid))\\\\n    return valid[0]\\\\n\\\\ndef setup(data,cache,work):\\\\n    work.mkdir(parents=True,exist_ok=True)\\\\n    old=json.loads((cache/\\\\\'configuration.json\\\\\').read_text())\\\\n    assert old[\\\\\'revision\\\\\']==c.REVISION and old[\\\\\'dim\\\\\']==c.DIM and old[\\\\\'max_length\\\\\']==256\\\\n    for path,size in old[\\\\\'files\\\\\']:\\\\n        actual=data/\\\\\'train\\\\\'/Path(path).name\\\\n        assert actual.exists() and actual.stat().st_size==size,\\\\\'Training dataset does not match cache.\\\\\'\\\\n    config={\\\\\'version\\\\\':\\\\\'deadline-upgrade-v2\\\\\',\\\\\'seed\\\\\':SEED,\\\\\'train_n\\\\\':TRAIN_N,\\\\\'tune_n\\\\\':TUNE_N,\\\\\'evaluation_n\\\\\':EVAL_N,\\\\n      \\\\\'old_matcher_sha256\\\\\':s.sha256(cache/\\\\\'matcher.pkl\\\\\'),\\\\\'old_sample_sha256\\\\\':s.sha256(cache/\\\\\'sample.tsv\\\\\'),\\\\n      \\\\\'cache_manifest_sha256\\\\\':s.sha256(cache/\\\\\'encoding_complete.json\\\\\'),\\\\n      \\\\\'profiles\\\\\':PROFILES,\\\\\'features\\\\\':c.MATCH_FEATURES,\\\\n      \\\\\'core_sha256\\\\\':s.sha256(Path(c.__file__)),\\\\\'runner_sha256\\\\\':s.sha256(Path(s.__file__)),\\\\n      \\\\\'upgrade_sha256\\\\\':s.sha256(Path(__file__))}\\\\n    config=json.loads(json.dumps(config))\\\\n    path=work/\\\\\'upgrade_config.json\\\\\'\\\\n    if path.exists():assert json.loads(path.read_text())==config,\\\\\'Configuration changed. Use a new UPGRADE_WORK.\\\\\'\\\\n    else:c.atomic_json(path,config)\\\\n\\\\n\\\\ndef sample_data(data,cache,work):\\\\n    path=work/\\\\\'sample.pkl\\\\\'\\\\n    if path.exists():\\\\n        with path.open(\\\\\'rb\\\\\') as f:return pickle.load(f)\\\\n    exclude=set(pd.read_csv(cache/\\\\\'sample.tsv\\\\\',sep=\\\\\'\\\\\\\\t\\\\\',dtype=str,keep_default_na=False).entity_id)\\\\n    rng=np.random.default_rng(SEED);sample=pd.DataFrame();n=TRAIN_N+TUNE_N+EVAL_N\\\\n    for chunk in c.read_chunks(data/\\\\\'train/train_source1.tsv\\\\\'):\\\\n        chunk=chunk.loc[~chunk.entity_id.isin(exclude),c.COLS].copy()\\\\n        chunk[\\\\\'_priority\\\\\']=rng.random(len(chunk))\\\\n        sample=pd.concat([sample,chunk],ignore_index=True).nsmallest(n,\\\\\'_priority\\\\\')\\\\n    sample=sample.drop(columns=\\\\\'_priority\\\\\').reset_index(drop=True)\\\\n    assert len(sample)==n and sample.entity_id.is_unique\\\\n    assert not set(sample.entity_id)&exclude\\\\n    ids=set(sample.entity_id);truth={}\\\\n    for chunk in c.read_chunks(data/\\\\\'train/train_ground_truth.tsv\\\\\',[\\\\\'source1_entity_id\\\\\',\\\\\'matched_entity_ids\\\\\']):\\\\n        for r in chunk[chunk.source1_entity_id.isin(ids)].itertuples(index=False):\\\\n            truth[r.source1_entity_id]=set(x for x in r.matched_entity_ids.split(\\\\\',\\\\\') if x)\\\\n    assert set(truth)==ids\\\\n    strata=sample.country+\\\\\'|\\\\\'+sample.entity_id.map(lambda x:str(not truth[x]))\\\\n    _,held=train_test_split(np.arange(n),test_size=TUNE_N+EVAL_N,random_state=SEED,stratify=strata)\\\\n    tune,evaluation=train_test_split(held,test_size=EVAL_N,random_state=SEED+1,stratify=strata.iloc[held])\\\\n    sample[\\\\\'split\\\\\']=\\\\\'train\\\\\';sample.loc[tune,\\\\\'split\\\\\']=\\\\\'tune\\\\\';sample.loc[evaluation,\\\\\'split\\\\\']=\\\\\'evaluation\\\\\'\\\\n    sample.to_csv(work/\\\\\'sample.tsv\\\\\',sep=\\\\\'\\\\\\\\t\\\\\',index=False)\\\\n    with Path(str(path)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f:pickle.dump((sample,truth),f)\\\\n    os.replace(str(path)+\\\\\'.tmp\\\\\',path)\\\\n    return sample,truth\\\\n\\\\n\\\\ndef queries(sample,work,encoder):\\\\n    path=work/\\\\\'queries.npy\\\\\'\\\\n    if path.exists():return np.load(path)\\\\n    out=[]\\\\n    for start in range(0,len(sample),1000):\\\\n        out.append(encoder.encode(sample.iloc[start:start+1000]))\\\\n        c.log(f\\\\\'Query embeddings: {min(start+1000,len(sample)):,}/{len(sample):,}\\\\\')\\\\n    q=np.concatenate(out)\\\\n    with Path(str(path)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f:np.save(f,q)\\\\n    os.replace(str(path)+\\\\\'.tmp\\\\\',path)\\\\n    return q\\\\n\\\\n\\\\ndef pair_arrays(frame,truth,owners_global,ids,scores,records):\\\\n    xs=[];ys=[];owners=[]\\\\n    for local,left in enumerate(frame[c.COLS].itertuples(index=False,name=None)):\\\\n        for rank,rid in enumerate(ids[local]):\\\\n            if rid<0:continue\\\\n            right=records[int(rid)]\\\\n            assert c.norm(left[3])==c.norm(right[3])\\\\n            xs.append(c.features(left,right)+[float(scores[local,rank])])\\\\n            ys.append(int(right[0] in truth[left[0]]));owners.append(int(owners_global[local]))\\\\n    return dict(x=np.asarray(xs,np.float32).reshape(-1,len(c.MATCH_FEATURES)),\\\\n                y=np.asarray(ys,np.uint8),owners=np.asarray(owners,np.int32))\\\\n\\\\n\\\\ndef prepare_pairs(cache,work,sample,truth,q,profile,indices):\\\\n    npb,retr=PROFILES[profile]\\\\n    folder=work/f\\\\\'pairs_{profile}\\\\\';folder.mkdir(exist_ok=True)\\\\n    search=s.TestSearcher(cache,nprobe=npb,overfetch=retr);search.faiss.omp_set_num_threads(4)\\\\n    pieces=[];started=time.time()\\\\n    try:\\\\n        with threadpool_limits(limits=4):\\\\n            for start in range(0,len(indices),500):\\\\n                selected=indices[start:start+500];path=folder/f\\\\\'{start:07d}.npz\\\\\'\\\\n                if path.exists():\\\\n                    with np.load(path) as z:\\\\n                        assert np.array_equal(z[\\\\\'query_indices\\\\\'],selected)\\\\n                        piece={k:z[k].copy() for k in [\\\\\'x\\\\\',\\\\\'y\\\\\',\\\\\'owners\\\\\']}\\\\n                else:\\\\n                    frame=sample.iloc[selected].reset_index(drop=True)\\\\n                    ids,scores=search.search(frame,q[selected]);records=search.fetch(ids)\\\\n                    piece=pair_arrays(frame,truth,selected,ids,scores,records)\\\\n                    with Path(str(path)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f:np.savez(f,query_indices=selected,**piece)\\\\n                    os.replace(str(path)+\\\\\'.tmp\\\\\',path)\\\\n                    del records,ids,scores\\\\n                pieces.append(piece)\\\\n                if start==0 or (start+len(selected))%2500==0:\\\\n                    c.log(f\\\\\'{profile}: {start+len(selected):,}/{len(indices):,} businesses; {(time.time()-started)/60:.1f} min\\\\\')\\\\n    finally:search.close()\\\\n    return {k:np.concatenate([piece[k] for piece in pieces]) for k in [\\\\\'x\\\\\',\\\\\'y\\\\\',\\\\\'owners\\\\\']}\\\\n\\\\n\\\\ndef metrics(prob,pairs,selected,truth_counts,threshold):\\\\n    n=len(truth_counts);accepted=prob>=threshold\\\\n    predicted=np.bincount(pairs[\\\\\'owners\\\\\'][accepted],minlength=n)\\\\n    tp=np.bincount(pairs[\\\\\'owners\\\\\'][accepted & (pairs[\\\\\'y\\\\\']==1)],minlength=n)\\\\n    fp=predicted-tp;fn=truth_counts-tp\\\\n    denom=1.25*tp+fp+.25*fn\\\\n    score=np.divide(1.25*tp,denom,out=np.zeros(n,dtype=float),where=denom!=0)\\\\n    score[(truth_counts==0)&(predicted==0)]=1\\\\n    singleton=truth_counts[selected]==0\\\\n    return dict(macro_F05=float(score[selected].mean()),correct_accepted=int(tp[selected].sum()),\\\\n       incorrect_accepted=int(fp[selected].sum()),missed_true_links=int(fn[selected].sum()),\\\\n       singletons_correct=int(((predicted[selected]==0)&singleton).sum()),singletons=int(singleton.sum()))\\\\n\\\\n\\\\ndef tune(prob,pairs,selected,counts):\\\\n    trials=[dict(threshold=float(t),**metrics(prob,pairs,selected,counts,t)) for t in np.linspace(.05,.95,91)]\\\\n    return max(trials,key=lambda x:(x[\\\\\'macro_F05\\\\\'],x[\\\\\'threshold\\\\\'])),trials\\\\n\\\\n\\\\ndef retrieval(pairs,selected,counts):\\\\n    hits=np.bincount(pairs[\\\\\'owners\\\\\'],weights=pairs[\\\\\'y\\\\\'],minlength=len(counts))\\\\n    sizes=np.bincount(pairs[\\\\\'owners\\\\\'],minlength=len(counts))\\\\n    found=int(hits[selected].sum());total=int(counts[selected].sum())\\\\n    return dict(found=found,true_links=total,recall=found/total if total else 1,\\\\n                mean_candidates=float(sizes[selected].mean()),max_candidates=int(sizes[selected].max()))\\\\n\\\\n\\\\ndef benchmark(cache,sample,encoder,bundle,profile):\\\\n    # Raw text -> embedding -> search -> lexical features -> model score, not search alone.\\\\n    npb,over=PROFILES[profile];search=s.TestSearcher(cache,npb,over)\\\\n    frame=sample.sample(n=min(1000,len(sample)),random_state=42).reset_index(drop=True)\\\\n    elapsed=[]\\\\n    try:\\\\n        with threadpool_limits(limits=4):\\\\n            search.faiss.omp_set_num_threads(4)\\\\n            for start in range(0,len(frame),250):\\\\n                part=frame.iloc[start:start+250].reset_index(drop=True)\\\\n                t=time.time();q=encoder.encode(part);ids,score=search.search(part,q)\\\\n                records=search.fetch(ids);s.score_candidates(part,ids,score,records,bundle)\\\\n                elapsed.append(time.time()-t)\\\\n    finally:search.close()\\\\n    return len(frame)/sum(elapsed)\\\\n\\\\n\\\\ndef run(data,cache,work,target_finish,batch_size=64):\\\\n    data=Path(data);cache=Path(cache);work=Path(work)\\\\n    setup(data,cache,work)\\\\n    old_bundle,old_meta=s.load_model(cache)\\\\n    sample,truth=sample_data(data,cache,work)\\\\n    c.log(\\\\\'Fresh sample: \\\\\'+str(sample.split.value_counts().to_dict()))\\\\n    encoder=c.Encoder(batch_size)\\\\n    q=queries(sample,work,encoder)\\\\n    strong=prepare_pairs(cache,work,sample,truth,q,\\\\\'strong\\\\\',np.arange(len(sample)))\\\\n    held=np.flatnonzero(sample.split.to_numpy()!=\\\\\'train\\\\\')\\\\n    standard=prepare_pairs(cache,work,sample,truth,q,\\\\\'standard\\\\\',held)\\\\n    train_mask=sample.split.to_numpy()[strong[\\\\\'owners\\\\\']]==\\\\\'train\\\\\'\\\\n    saved=work/\\\\\'new_classifier.pkl\\\\\'\\\\n    if saved.exists():\\\\n        with saved.open(\\\\\'rb\\\\\') as f:model=pickle.load(f)\\\\n    else:\\\\n        c.log(f\\\\\'Training 30k-business matcher: {train_mask.sum():,} pairs; {strong[\\"y\\"][train_mask].sum():,} positives\\\\\')\\\\n        model=HistGradientBoostingClassifier(learning_rate=.08,max_iter=250,max_leaf_nodes=31,\\\\n                    l2_regularization=1.,early_stopping=False,random_state=2026)\\\\n        with threadpool_limits(limits=4):model.fit(strong[\\\\\'x\\\\\'][train_mask],strong[\\\\\'y\\\\\'][train_mask])\\\\n        with Path(str(saved)+\\\\\'.tmp\\\\\').open(\\\\\'wb\\\\\') as f:pickle.dump(model,f)\\\\n        os.replace(str(saved)+\\\\\'.tmp\\\\\',saved)\\\\n    strong={k:a[~train_mask] for k,a in strong.items()}\\\\n    pairs={\\\\\'standard\\\\\':standard,\\\\\'strong\\\\\':strong}\\\\n    counts=np.array([len(truth[x]) for x in sample.entity_id])\\\\n    ti=np.flatnonzero(sample.split.to_numpy()==\\\\\'tune\\\\\');ei=np.flatnonzero(sample.split.to_numpy()==\\\\\'evaluation\\\\\')\\\\n    scores={};options={};benchmark_rates={};all_trials=[]\\\\n    total_test=sum(len(chunk) for chunk in c.read_chunks(data/\\\\\'test/test_source1.tsv\\\\\',[\\\\\'entity_id\\\\\']))\\\\n    for name,arr in pairs.items():\\\\n        with threadpool_limits(limits=4):scores[name]=model.predict_proba(arr[\\\\\'x\\\\\'])[:,1]\\\\n        best,trials=tune(scores[name],arr,ti,counts);options[name]=best\\\\n        all_trials.extend(dict(profile=name,**r) for r in trials)\\\\n        bundle={\\\\\'model\\\\\':model,\\\\\'threshold\\\\\':best[\\\\\'threshold\\\\\']}\\\\n        benchmark_rates[name]=benchmark(cache,sample,encoder,bundle,name)\\\\n        c.log(f\\\\\'{name}: tuning macro F0.5={best[\\"macro_F05\\"]:.5f}; prediction benchmark={benchmark_rates[name]:.1f} businesses/sec\\\\\')\\\\n    # Profile and threshold chosen using tuning scores and time budget, before evaluation.\\\\n    finish=datetime.fromisoformat(target_finish)\\\\n    assert finish.tzinfo is not None,\\\\\'Use an ISO timestamp with timezone, e.g. +05:30.\\\\\'\\\\n    remaining_hours=(finish-datetime.now(timezone.utc)).total_seconds()/3600\\\\n    # 35% slower than warm training benchmark, plus 2h test encoding/index/export allowance.\\\\n    estimates={name:total_test/rate/3600*1.35+2.0 for name,rate in benchmark_rates.items()}\\\\n    feasible=[name for name in pairs if estimates[name]<=remaining_hours]\\\\n    chosen=max(feasible or list(pairs),key=lambda name:(options[name][\\\\\'macro_F05\\\\\'],name==\\\\\'standard\\\\\'))\\\\n    cutoff=options[chosen][\\\\\'threshold\\\\\']\\\\n    with threadpool_limits(limits=4):old_prob=old_bundle[\\\\\'model\\\\\'].predict_proba(standard[\\\\\'x\\\\\'])[:,1]\\\\n    baseline=metrics(old_prob,standard,ei,counts,old_bundle[\\\\\'threshold\\\\\'])\\\\n    evaluation=metrics(scores[chosen],pairs[chosen],ei,counts,cutoff)\\\\n    search_eval=retrieval(pairs[chosen],ei,counts)\\\\n    evaluation[\\\\\'missed_by_search\\\\\']=search_eval[\\\\\'true_links\\\\\']-search_eval[\\\\\'found\\\\\']\\\\n    evaluation[\\\\\'retrieved_but_rejected\\\\\']=evaluation[\\\\\'missed_true_links\\\\\']-evaluation[\\\\\'missed_by_search\\\\\']\\\\n    # Only one fresh evaluation comparison; retain the already-submitted fallback on regression.\\\\n    promote=bool(feasible and evaluation[\\\\\'macro_F05\\\\\']>baseline[\\\\\'macro_F05\\\\\'])\\\\n    report={\\\\\'old_model_fresh_evaluation\\\\\':baseline,\\\\\'new_model_fresh_evaluation\\\\\':evaluation,\\\\n       \\\\\'tuning_options\\\\\':options,\\\\\'chosen_profile\\\\\':chosen,\\\\\'threshold\\\\\':cutoff,\\\\n       \\\\\'search_recall_evaluation\\\\\':search_eval,\\\\\'prediction_benchmark_per_second\\\\\':benchmark_rates,\\\\n       \\\\\'estimated_test_hours_with_margin\\\\\':estimates,\\\\\'hours_until_target_finish\\\\\':remaining_hours,\\\\n       \\\\\'target_finish\\\\\':target_finish,\\\\\'promote_to_test\\\\\':promote,\\\\n       \\\\\'note\\\\\':\\\\\'Timing is an estimate, not a guarantee. Evaluation excludes all previous 4000 sampled S1 records.\\\\\'}\\\\n    c.atomic_json(work/\\\\\'upgrade_report.json\\\\\',report)\\\\n    pd.DataFrame(all_trials).to_csv(work/\\\\\'threshold_tuning.csv\\\\\',index=False)\\\\n    by_country=[]\\\\n    for country in sorted(sample.country.unique()):\\\\n        selected=ei[sample.country.to_numpy()[ei]==country]\\\\n        by_country.append({\\\\\'country\\\\\':country,\\\\\'businesses\\\\\':len(selected),\\\\n           \\\\\'old\\\\\':metrics(old_prob,standard,selected,counts,old_bundle[\\\\\'threshold\\\\\']),\\\\n           \\\\\'new\\\\\':metrics(scores[chosen],pairs[chosen],selected,counts,cutoff)})\\\\n    c.atomic_json(work/\\\\\'country_evaluation.json\\\\\',by_country)\\\\n    model_dir=work/\\\\\'model\\\\\';model_dir.mkdir(exist_ok=True)\\\\n    bundle={\\\\\'model\\\\\':model,\\\\\'threshold\\\\\':cutoff,\\\\\'features\\\\\':c.MATCH_FEATURES,\\\\n            \\\\\'ann_nprobe\\\\\':PROFILES[chosen][0],\\\\\'ann_overfetch\\\\\':PROFILES[chosen][1]}\\\\n    with (model_dir/\\\\\'matcher.pkl\\\\\').open(\\\\\'wb\\\\\') as f:pickle.dump(bundle,f)\\\\n    from importlib.metadata import version\\\\n    meta={\\\\\'version\\\\\':\\\\\'deadline-upgrade-v2\\\\\',\\\\\'model_id\\\\\':c.MODEL_ID,\\\\\'revision\\\\\':c.REVISION,\\\\\'candidate_k\\\\\':20,\\\\n       \\\\\'ann_nprobe\\\\\':PROFILES[chosen][0],\\\\\'ann_overfetch\\\\\':PROFILES[chosen][1],\\\\\'threshold\\\\\':cutoff,\\\\n       \\\\\'evaluation\\\\\':evaluation,\\\\\'tuning\\\\\':options[chosen],\\\\\'features\\\\\':c.MATCH_FEATURES,\\\\n       \\\\\'train_businesses\\\\\':TRAIN_N,\\\\\'tune_businesses\\\\\':TUNE_N,\\\\\'evaluation_businesses\\\\\':EVAL_N,\\\\n       \\\\\'versions\\\\\':{p:version(p) for p in [\\\\\'numpy\\\\\',\\\\\'pandas\\\\\',\\\\\'scikit-learn\\\\\',\\\\\'faiss-cpu\\\\\',\\\\\'torch\\\\\',\\\\\'transformers\\\\\']}}\\\\n    c.atomic_json(model_dir/\\\\\'experiment_results.json\\\\\',meta)\\\\n    print(json.dumps(report,indent=2),flush=True)\\\\n    print(\\\\\'AUTO TEST RUN APPROVED\\\\\' if promote else \\\\\'KEEP 0.84 FALLBACK: validation did not improve or estimated runtime exceeds target.\\\\\',flush=True)\\\\n    del encoder;gc.collect()\\\\n    import torch;torch.cuda.empty_cache()\\\\n    return report\\\\n\\\\nif __name__==\\\\\'__main__\\\\\':\\\\n    ap=argparse.ArgumentParser();ap.add_argument(\\\\\'--data\\\\\',type=Path,required=True)\\\\n    ap.add_argument(\\\\\'--cache\\\\\',type=Path,required=True);ap.add_argument(\\\\\'--work\\\\\',type=Path,required=True)\\\\n    ap.add_argument(\\\\\'--target-finish\\\\\',required=True);ap.add_argument(\\\\\'--batch-size\\\\\',type=int,default=64)\\\\n    args=ap.parse_args();run(args.data,args.cache,args.work,args.target_finish,args.batch_size)\\\\n\'}\\n",\n    "for name,text in MODULES.items():\\n",\n    "    (CODE_DIR/name).write_text(text)\\n",\n    "probe=subprocess.run([sys.executable,\'-c\',\\n",\n    "    \'import upgrade_train as u; print(u.resolve_cache(\'+repr(CACHE_DIR or None)+\'))\'],\\n",\n    "    cwd=CODE_DIR,text=True,capture_output=True)\\n",\n    "if probe.returncode:\\n",\n    "    print(probe.stderr)\\n",\n    "    raise RuntimeError(\'Attach the FULL saved previous notebook output and check CACHE_DIR.\')\\n",\n    "CACHE_DIR=Path(probe.stdout.strip())\\n",\n    "print(\'Read-only training cache:\',CACHE_DIR)\\n",\n    "print(\'Training target vectors will NOT be regenerated or copied.\')\\n",\n    "\\n",\n    "def run_logged(command,name):\\n",\n    "    with (ROOT/name).open(\'a\') as log:\\n",\n    "        proc=subprocess.Popen(command,cwd=CODE_DIR,stdout=subprocess.PIPE,\\n",\n    "            stderr=subprocess.STDOUT,text=True,bufsize=1)\\n",\n    "        for line in proc.stdout:\\n",\n    "            print(line,end=\'\',flush=True);log.write(line);log.flush()\\n",\n    "        returncode=proc.wait()\\n",\n    "    if returncode:\\n",\n    "        raise RuntimeError(\'Stage stopped. See the preceding log. Completed checkpoints remain in ROOT.\')\\n"\n   ],\n   "outputs": [],\n   "execution_count": null,\n   "id": "training-3"\n  },\n  {\n   "cell_type": "markdown",\n   "metadata": {},\n   "source": [\n    "## 1. Train and compare using the saved training pool\\n",\n    "This draws a new sample excluding **all 4,000 businesses from the old experiment**. It generates real retrieval candidates, without injecting ground-truth matches. Training and pair-feature batches are checkpointed.\\n",\n    "\\n",\n    "The old model is scored using its original cutoff and standard search. The new model\'s cutoff and search profile use only the tuning businesses. Both are then compared on the same fresh evaluation businesses. This is a local comparison, not a guarantee of a leaderboard gain; France remains absent from labeled training.\\n",\n    "\\n",\n    "The runtime benchmark includes query encoding, retrieval, feature computation, and classifier prediction. Its forecast adds 35% to benchmarked prediction time plus two hours for test encoding/indexing/export. Cold disk access and GPU variation can still make the full run slower.\\n"\n   ],\n   "id": "training-4"\n  },\n  {\n   "cell_type": "code",\n   "metadata": {},\n   "source": [\n    "run_logged([sys.executable,str(CODE_DIR/\'upgrade_train.py\'),\\n",\n    "    \'--data\',str(DATA),\'--cache\',str(CACHE_DIR),\'--work\',str(UPGRADE_WORK),\\n",\n    "    \'--target-finish\',TARGET_FINISH,\'--batch-size\',str(BATCH_SIZE)],\'upgrade_training.log\')\\n",\n    "report=json.loads((UPGRADE_WORK/\'upgrade_report.json\').read_text())\\n",\n    "print(json.dumps(report,indent=2))\\n"\n   ],\n   "outputs": [],\n   "execution_count": null,\n   "id": "training-5"\n  }\n ],\n "metadata": {\n  "kernelspec": {\n   "display_name": "Python 3",\n   "language": "python",\n   "name": "python3"\n  },\n  "language_info": {\n   "name": "python"\n  }\n },\n "nbformat": 4,\n "nbformat_minor": 5\n}'
from datetime import datetime,timezone
report=json.loads((UPGRADE_WORK/'upgrade_report.json').read_text())
selected=report['chosen_profile']
remaining=(datetime.fromisoformat(TARGET_FINISH)-datetime.now(timezone.utc)).total_seconds()/3600
needed=report['estimated_test_hours_with_margin'][selected]
RUN_SUBMISSION=report['promote_to_test'] and remaining>=needed
if RUN_SUBMISSION:
    SUBMISSION_WORK.mkdir(parents=True,exist_ok=True)
    for name,text in MODULES.items():
        (SUBMISSION_WORK/name).write_text(text)
    (SUBMISSION_WORK/'training_experiment.ipynb').write_text(TRAINING_NOTEBOOK)
    run_logged([sys.executable,str(SUBMISSION_WORK/'submit_pipeline.py'),
        '--data',str(DATA),'--work',str(SUBMISSION_WORK),
        '--model-dir',str(UPGRADE_WORK/'model'),'--stage','all',
        '--batch-size',str(BATCH_SIZE)],'test_submission.log')
else:
    print('Full test run NOT started. Keep the existing 0.84 submission.')
    print('Evaluation gate:',report['promote_to_test'])
    print('Hours remaining until target:',round(remaining,2),'Estimated hours needed:',round(needed,2))


## 3. Download results
Only upload the new **matching_results.tsv** after the official validator passes. If a TSV opens a preview, download its ZIP and extract it. Keep the candidate file and backup for final review.

If the evaluation/time gate did not pass, you still get the training report and saved new model for inspection, but your existing 0.84 submission remains the fallback. No leaderboard submission is changed automatically.


In [ ]:
import zipfile
from IPython.display import FileLink,display
# Small upgrade report backup, independent of whether test prediction ran.
report_zip=ROOT/'upgrade_training_results.zip'
with zipfile.ZipFile(report_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['upgrade_config.json','upgrade_report.json','country_evaluation.json',
                 'threshold_tuning.csv','sample.tsv','model/matcher.pkl','model/experiment_results.json']:
        if (UPGRADE_WORK/name).exists():z.write(UPGRADE_WORK/name,name)
    for name in MODULES:z.write(CODE_DIR/name,name)
    z.writestr('training_experiment.ipynb',TRAINING_NOTEBOOK)
display(FileLink(str(report_zip)))
out=SUBMISSION_WORK/'output'
if (out/'validated.json').exists():
    print('VALIDATED — upload matching_results.tsv:')
    display(FileLink(str(out/'matching_results.tsv')))
    display(FileLink(str(out/'matching_results_download.zip')))
    print('Retain for final review:')
    display(FileLink(str(out/'candidate_pairs.tsv')))
    display(FileLink(str(ROOT/'embedding_submission_v2_backup.zip')))
else:
    print('No validated replacement submission yet. Your existing 0.84 remains the fallback.')
